<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇸🇪 Sweden Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: trafiklab.se · Provider: Samtrafiken (Swedish National Access Point) · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

Both datasets were obtained from **Trafiklab** (`trafiklab.se`), the Swedish national
access point for public transport data, operated by Samtrafiken on behalf of the
regional public transport authorities in Sweden.

To access the data, a free account was created on the Trafiklab developer portal
(`developer.trafiklab.se`). A project was set up under the type **Educational project**,
and API keys were requested for both feeds. Keys are approved instantly at Bronze level,
which allows up to 50 downloads per month.

The two feeds used are:

**GTFS Sverige 2**
- Full national coverage of all public transport in Sweden
- Downloaded as a single static ZIP file via:
  `https://api.resrobot.se/gtfs/sweden.zip?key={apikey}`
- License: CC0 1.0 (Public Domain)

**NeTEx Sweden Static data**
- National NeTEx feed covering the same operators as GTFS Sweden
- Downloaded as a single static ZIP file via:
  `https://opendata.samtrafiken.se/netex-sweden/sweden.zip?key={apikey}`
- License: CC0 1.0 (Public Domain)

Both feeds are updated daily and are free to use without restrictions.

> **_NOTE:_** the NeTEx Sweden feed does not yet cover all operators in Sweden,
while GTFS Sverige 2 provides full national coverage. Where operators appear
in NeTEx but not GTFS or vice versa, this will be noted in the comparison.

## Table of Contents

**Data Source**

**Part 1: GTFS Exploration**
- Inspect GTFS stop types
- Check GTFS stop data quality
- Inspect GTFS route labels
- Inspect routes without route_short_name
- Inspect examples of routes without route_short_name
- Check route_long_name as possible fallback label
- Check route_long_name completeness
- Inspect GTFS route types
- Create GTFS public line label
- Inspect GTFS stop names and coordinates
- Inspect GTFS calendar coverage
- Inspect GTFS calendar date ranges and exceptions
- Inspect GTFS weekly calendar patterns
- Inspect GTFS active dates from calendar_dates
- Inspect distribution of GTFS active date counts
- Create GTFS calendar summary for comparison
- Check GTFS trips as bridge table
- Inspect missing service_id from calendar summary
- Summary of GTFS exploration

**Part 2: NeTEx Exploration**
- Inspect NeTEx file structure
- Inspect sample StopPlace and Quay records
- Extract all NeTEx StopPlaces
- Data quality check on the extracted StopPlaces
- Inspect NeTEx line structure
- Extract all NeTEx Lines
- Prepare NeTEx line-level dataset
- Inspect NeTEx calendar structure
- Extract NeTEx calendar data
- Build NeTEx active dates
- Prepare NeTEx calendar summary
- Summary of NeTEx exploration

**Part 3: GTFS–NeTEx Comparison**
- 3.1 Station-level comparison
- 3.2 Line-level comparison
- 3.3 Calendar-level comparison
  - Calendar ID overlap check
  - Pattern-level calendar comparison
  - Build and compare calendar activity patterns
  - Compare unique calendar activity patterns
  - Calendar comparison: approach and results
  - Overall summary

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import re
import xml.etree.ElementTree as ET
from scipy.spatial import cKDTree
import numpy as np

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "GTFS Sverige 2.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/GTFS Sverige 2.zip


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
5,stop_times.txt,537.33
8,trips.txt,28.47
7,transfers.txt,7.92
0,calendar_dates.txt,3.98
6,stops.txt,2.16
4,routes.txt,0.27
1,calendar.txt,0.26
3,feed_info.txt,0.00
2,agency.txt,0.00


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:", stops.shape)
print("routes:", routes.shape)
print("trips:", trips.shape)
print("calendar:", calendar.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (48778, 5)
routes: (9153, 6)
trips: (523669, 5)
calendar: (6893, 10)
calendar_dates: (231792, 3)


In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type']


,stop_id,stop_name,stop_lat,stop_lon,location_type
0,100000351,Tornio På Gränsen Rajala,65.843295,24.145137,NaN
1,100083169,Lappia Yrkesskola (FIN),65.851576,24.136418,NaN
2,100083170,Torneå Badhus (FIN),65.839902,24.177822,NaN
3,100083171,Torppi Handelsområde (FIN),65.843543,24.183866,NaN
4,510000027,Gorzow Wielkopolski,52.728473,15.227841,NaN


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_url']


,route_id,agency_id,route_short_name,route_long_name,route_type,route_url
0,1074000100001,74,NaN,SJ Nattåg,102,NaN
1,1074000200001,74,NaN,SJ Nattåg,102,NaN
2,1074001000001,74,NaN,SJ InterCity,102,NaN
3,1074001100001,74,NaN,SJ InterCity,102,NaN
4,1074001200001,74,NaN,SJ InterCity,102,NaN


trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name']


,route_id,service_id,trip_id,trip_headsign,trip_short_name
0,1258012000001,681,22580120000001,Jämjö Parkvägen,NaN
1,1258012000001,513,22580120010001,Jämjö Parkvägen,NaN
2,1258012000001,492,22580120000003,Jämjö Parkvägen,NaN
3,1258012000001,26,22580120010002,Jämjö Parkvägen,NaN
4,1258012000001,243,22580120010003,Jämjö Parkvägen,NaN


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,0,0,0,0,0,0,0,20260501,20270430
1,3,0,0,0,0,0,0,0,20260501,20270430
2,5,0,0,0,0,0,0,0,20260501,20270430
3,7,0,0,0,0,0,0,0,20260501,20270430
4,8,0,0,0,0,0,0,0,20260501,20270430


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,1,20260509,1
1,1,20260516,1
2,1,20260523,1
3,1,20260530,1
4,1,20260613,1


## Comment


The Swedish GTFS feed contains the main files needed:

- `stops.txt`, `routes.txt`, `trips.txt`, `calendar.txt`, and `calendar_dates.txt` are available
- The stop table has no `parent_station` column, this is different from what was seen in Norway
- The stop table only contains `stop_id`, `stop_name`, coordinates, and `location_type`
- The route table contains `route_short_name`, but the first rows already show some missing values
- Both calendar files are available, so the GTFS calendar logic can probably be rebuilt later from `calendar.txt` plus `calendar_dates.txt`

For the next step, the most important thing is to inspect `location_type`, because Sweden may not have the same station/quay hierarchy as Norway

## Inspect GTFS stop types

The Swedish `stops.txt` file does not contain a `parent_station` column.  
This means that the Norway logic with station-level rows and quay-level rows cannot be reused directly.

Therefore, I first inspect the `location_type` values.  
This helps to understand whether the feed separates stations from stops, or whether most rows are simple stop records.

In [6]:
# Inspect location_type values in the Swedish GTFS stops table

location_type_summary = (
    stops["location_type"]
    .fillna("missing")
    .value_counts(dropna=False)
    .reset_index()
)

location_type_summary.columns = ["location_type", "n_rows"]
location_type_summary["share_percent"] = (
    location_type_summary["n_rows"] / len(stops) * 100
).round(2)

location_type_summary

,location_type,n_rows,share_percent
0,missing,48778,100.0


## Observation

All rows in the Swedish `stops.txt` file have a missing `location_type`.

This means that the GTFS feed does not separate stations and platforms in the same way as Norway.  
For Sweden, the GTFS stop-level comparison will probably have to use all stop rows as the stop/station-level unit.

## Check GTFS stop data quality

Since all `location_type` values are missing, the Swedish GTFS feed does not clearly separate stations and platforms.

For this reason, I first treat all rows in `stops.txt` as GTFS stop-level records.  
Before using them for comparison, I check whether the important fields are complete:

- stop ID
- stop name
- latitude
- longitude

I also check whether stop IDs are unique.

In [7]:
# Basic quality check for Swedish GTFS stops

gtfs_stop_quality = pd.DataFrame({
    "check": [
        "number of stop rows",
        "unique stop_id values",
        "missing stop_id",
        "missing stop_name",
        "missing stop_lat",
        "missing stop_lon",
        "duplicate stop_id values"
    ],
    "value": [
        len(stops),
        stops["stop_id"].nunique(),
        stops["stop_id"].isna().sum(),
        stops["stop_name"].isna().sum(),
        stops["stop_lat"].isna().sum(),
        stops["stop_lon"].isna().sum(),
        stops["stop_id"].duplicated().sum()
    ]
})

gtfs_stop_quality

,check,value
0,number of stop rows,48778
1,unique stop_id values,48778
2,missing stop_id,0
3,missing stop_name,0
4,missing stop_lat,0
5,missing stop_lon,0
6,duplicate stop_id values,0


## Observation

The Swedish GTFS stop table is clean.

- there are 48,778 stop rows
- all `stop_id` values are unique
- there are no missing stop IDs, stop names, coordinates
- there are no duplicate stop IDs

Even though `location_type` is missing, the stop table itself is complete enough to continue with the stop-level comparison later.

## Inspect GTFS route labels

After checking the stop table, I now inspect the GTFS route table.

For the line-level MVD, the most important field is `route_short_name`, because it usually represents the public line label.

I check:

- how many route rows exist
- how many routes have a `route_short_name`
- how many are missing it
- how many unique public line labels exist
- whether several route rows share the same public label

In [8]:
# Check GTFS route_short_name values

routes_check = routes.copy()

# Keep route labels as strings
routes_check["route_short_name"] = routes_check["route_short_name"].astype("string")

gtfs_route_quality = pd.DataFrame({
    "check": [
        "number of route rows",
        "routes with route_short_name",
        "routes missing route_short_name",
        "unique route_short_name values",
        "duplicate route_short_name rows"
    ],
    "value": [
        len(routes_check),
        routes_check["route_short_name"].notna().sum(),
        routes_check["route_short_name"].isna().sum(),
        routes_check["route_short_name"].dropna().nunique(),
        routes_check["route_short_name"].dropna().duplicated().sum()
    ]
})

gtfs_route_quality

,check,value
0,number of route rows,9153
1,routes with route_short_name,6038
2,routes missing route_short_name,3115
3,unique route_short_name values,2870
4,duplicate route_short_name rows,3168


## Observation

The Swedish GTFS route table is usable, but less complete than Norway.

- There are 9,153 route rows
- 6,038 routes have a `route_short_name`
- 3,115 routes are missing `route_short_name`
- There are 2,870 unique public line labels
- Many route rows share the same `route_short_name`

This probably means that deduplication by public line label will be necessary later.  

## Inspect routes without route_short_name

The previous check showed that many Swedish GTFS routes do not have a `route_short_name`.

Before deciding the final line-level comparison rule, I inspect these missing rows more closely.  
This helps to see whether the missing labels belong to specific transport modes or service types.

This is important because the line comparison should use public-facing line labels, not technical route rows.

In [9]:
# Inspect routes where route_short_name is missing

missing_route_short_name = routes_check[routes_check["route_short_name"].isna()].copy()

# Count missing route_short_name rows by route_type
missing_by_route_type = (
    missing_route_short_name["route_type"]
    .value_counts(dropna=False)
    .reset_index()
)

missing_by_route_type.columns = ["route_type", "n_missing_routes"]

missing_by_route_type

,route_type,n_missing_routes
0,106,1885
1,102,617
2,101,424
3,702,178
4,1000,5
5,900,4
6,700,2


## Observation

The missing `route_short_name` values are not spread evenly across all route types:

- Most missing labels are in route type `106`
- Route types `102` and `101` also have many missing labels
- Other route types have much fewer missing values

This suggests that missing `route_short_name` values are linked to specific transport modes or service types.  


## Inspect examples of routes without route_short_name

The previous result showed that many missing `route_short_name` values belong to specific route types.

Now I inspect example rows where `route_short_name` is missing.  
This helps to understand whether these routes still have useful public information in `route_long_name`.

In [10]:
# Show example routes where route_short_name is missing

missing_route_short_name_examples = (
    missing_route_short_name[
        ["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]
    ]
    .sort_values(["route_type", "route_long_name"])
    .head(30)
)

missing_route_short_name_examples

,route_id,agency_id,route_short_name,route_long_name,route_type
6539,1287000000001,287,<NA>,Arlanda Express,101
6540,1287000000002,287,<NA>,Arlanda Express,101
6541,1287000000003,287,<NA>,Arlanda Express,101
6542,1287000000004,287,<NA>,Arlanda Express,101
6543,1287000000005,287,<NA>,Arlanda Express,101
6544,1287000000006,287,<NA>,Arlanda Express,101
6545,1287000000007,287,<NA>,Arlanda Express,101
6546,1287000000008,287,<NA>,Arlanda Express,101
6547,1287000000009,287,<NA>,Arlanda Express,101
6548,1287000000010,287,<NA>,Arlanda Express,101


## Observation

The example rows show that missing `route_short_name` does not mean that the route has no public information.

- These routes have no `route_short_name`
- However, they do have a clear `route_long_name`
- In this sample, many rows refer to `Arlanda Express`
- The same public service appears several times as separate technical route rows

This means that `route_long_name` may be useful as a fallback label for some Swedish routes.  
It also confirms again that raw route rows should not be compared directly without deduplication.

## Check route_long_name as possible fallback label

The example rows showed that some routes without `route_short_name` still have useful public information in `route_long_name`.

Now I check the most common `route_long_name` values among routes with missing `route_short_name`.

This helps to decide whether `route_long_name` can be used later as a fallback label for the line-level comparison.

In [11]:
# Most common route_long_name values among routes missing route_short_name

missing_route_long_name_counts = (
    missing_route_short_name["route_long_name"]
    .value_counts(dropna=False)
    .reset_index()
)

missing_route_long_name_counts.columns = ["route_long_name", "n_rows"]

missing_route_long_name_counts.head(20)

,route_long_name,n_rows
0,Västtågen,909
1,Pågatågen,564
2,Krösatåg,287
3,Öresundståg,287
4,Arlanda Express,241
5,Östgötapendeln,143
6,SJ Regionaltåg,134
7,SJ Snabbtåg X2000,108
8,Vy Express,108
9,SJ InterCity,97


## Observation

The missing `route_short_name` values are linked to many recognizable services.

- Many missing labels are train services, such as `Västtågen`, `Pågatågen`, `Öresundståg`, and `Krösatåg`
- `Arlanda Express` also appears often. This is the fast airport train between Arlanda Airport and Stockholm
- These routes do not have `route_short_name`, but `route_long_name` gives useful public-facing information
- Several services appear many times as separate technical route rows

This suggests that for Sweden, `route_long_name` may be needed as a fallback label when `route_short_name` is missing.

## Check route_long_name completeness

In [12]:
# Check completeness of route_long_name

gtfs_route_long_name_quality = pd.DataFrame({
    "check": [
        "number of route rows",
        "routes with route_long_name",
        "routes missing route_long_name",
        "unique route_long_name values",
        "duplicate route_long_name rows"
    ],
    "value": [
        len(routes_check),
        routes_check["route_long_name"].notna().sum(),
        routes_check["route_long_name"].isna().sum(),
        routes_check["route_long_name"].dropna().nunique(),
        routes_check["route_long_name"].dropna().duplicated().sum()
    ]
})

gtfs_route_long_name_quality

,check,value
0,number of route rows,9153
1,routes with route_long_name,3115
2,routes missing route_long_name,6038
3,unique route_long_name values,34
4,duplicate route_long_name rows,3081


## Observation

`route_long_name` is not complete for all Swedish routes

- 3,115 routes have a `route_long_name`
- 6,038 routes are missing `route_long_name`
- This is the opposite of `route_short_name`: routes without `route_short_name` often have `route_long_name`
- There are only 34 unique `route_long_name` values, so many technical route rows share the same long name

This supports using `route_long_name` only as a fallback, not as the main line label for all routes.

## Inspect GTFS route types

Before finishing the GTFS line exploration, I inspect the `route_type` values in the full route table.

This helps to understand which transport modes are included in the Swedish GTFS feed.  
It also gives context for the missing `route_short_name` values, because some route types may use `route_long_name` instead of a short public line label.

In [13]:
# Count route_type values in the full GTFS routes table

gtfs_route_type_summary = (
    routes_check["route_type"]
    .value_counts(dropna=False)
    .reset_index()
)

gtfs_route_type_summary.columns = ["route_type", "n_routes"]

gtfs_route_type_summary

,route_type,n_routes
0,700,3809
1,106,2139
2,102,1395
3,702,655
4,101,504
5,1501,475
6,1000,146
7,900,23
8,401,7


## Observation

The Swedish GTFS route table contains several different `route_type` values.

- The largest group is route type `700`
- Route types `106`, `102`, and `702` also appear often
- This shows that the Swedish GTFS feed is multimodal
- Some of the route types with many routes are also the ones where `route_short_name` was often missing

## Create GTFS public line label

For the Swedish GTFS feed, `route_short_name` and `route_long_name` seem to complement each other.

Many rows with `route_short_name` do not have `route_long_name`, and many rows without `route_short_name` do have `route_long_name`.

Therefore, I create one combined public line label:

- use `route_short_name` when available
- otherwise use `route_long_name`

This keeps the comparison close to the Norway workflow, but adapts it to the Swedish GTFS structure.

In [14]:
# Create one combined GTFS public line label for Sweden

routes_check["gtfs_public_label"] = routes_check["route_short_name"].fillna(
    routes_check["route_long_name"]
)

# Keep labels as strings
routes_check["gtfs_public_label"] = routes_check["gtfs_public_label"].astype("string")

gtfs_public_label_quality = pd.DataFrame({
    "check": [
        "number of route rows",
        "routes with gtfs_public_label",
        "routes missing gtfs_public_label",
        "unique gtfs_public_label values",
        "duplicate gtfs_public_label rows"
    ],
    "value": [
        len(routes_check),
        routes_check["gtfs_public_label"].notna().sum(),
        routes_check["gtfs_public_label"].isna().sum(),
        routes_check["gtfs_public_label"].dropna().nunique(),
        routes_check["gtfs_public_label"].dropna().duplicated().sum()
    ]
})

gtfs_public_label_quality

,check,value
0,number of route rows,9153
1,routes with gtfs_public_label,9153
2,routes missing gtfs_public_label,0
3,unique gtfs_public_label values,2904
4,duplicate gtfs_public_label rows,6249


In [15]:
fallback_count = routes_check["route_short_name"].isna().sum()
print(f"Routes using route_long_name as fallback: {fallback_count}")

Routes using route_long_name as fallback: 3115


## Observation

The combined GTFS public label works well for Sweden.

- All 9,153 route rows now have a public label
- 2,904 unique public labels are available
- 3,115 routes use `route_long_name` as a fallback because `route_short_name` is missing 
- Many route rows still share the same public label, so deduplication will be necessary later


>Because the Swedish GTFS feed does not contain `parent_station` and all `location_type` values are missing, the stop hierarchy cannot be reconstructed on the GTFS side.

>Therefore, the Swedish comparison cannot use the same station/platform separation as Norway.  
Instead, all GTFS stop rows are treated as stop-level records.

>The comparable NeTEx level will be decided after inspecting the Swedish NeTEx stop structure. If NeTEx Quays represent the detailed stop level, they will be used for comparison. If StopPlaces are closer to the GTFS stop records, StopPlaces will be used instead.

In [16]:
# Add readable transport mode names for the route_type values used in the Swedish GTFS feed

route_type_names = {
    101: "high-speed rail",
    102: "long-distance rail",
    106: "regional rail",
    401: "metro / urban rail",
    700: "bus",
    702: "express bus",
    900: "tram",
    1000: "water transport",
    1501: "taxi / demand-responsive transport"
}

gtfs_route_type_summary["transport_mode"] = gtfs_route_type_summary["route_type"].map(route_type_names)

gtfs_route_type_summary

,route_type,n_routes,transport_mode
0,700,3809,bus
1,106,2139,regional rail
2,102,1395,long-distance rail
3,702,655,express bus
4,101,504,high-speed rail
5,1501,475,taxi / demand-responsive transport
6,1000,146,water transport
7,900,23,tram
8,401,7,metro / urban rail


## Observation

The Swedish GTFS feed is clearly multimodal:

- The largest group is bus routes
- Regional rail and long-distance rail are also strongly represented
- The feed also includes express bus, high-speed rail, taxi / demand-responsive transport, water transport, tram, and metro / urban rail
- This shows that Sweden has a broad mix of transport services in GTFS


## Inspect GTFS stop names and coordinates


In [17]:
# Check repeated stop names and coordinate pairs in Swedish GTFS stops

gtfs_stop_repetition_check = pd.DataFrame({
    "check": [
        "number of stop rows",
        "unique stop_name values",
        "duplicate stop_name rows",
        "unique coordinate pairs",
        "duplicate coordinate pair rows"
    ],
    "value": [
        len(stops),
        stops["stop_name"].nunique(),
        stops["stop_name"].duplicated().sum(),
        stops[["stop_lat", "stop_lon"]].drop_duplicates().shape[0],
        stops[["stop_lat", "stop_lon"]].duplicated().sum()
    ]
})

gtfs_stop_repetition_check

,check,value
0,number of stop rows,48778
1,unique stop_name values,44421
2,duplicate stop_name rows,4357
3,unique coordinate pairs,48744
4,duplicate coordinate pair rows,34


In [18]:
# Show examples of repeated stop names

repeated_stop_names = (
    stops["stop_name"]
    .value_counts()
    .reset_index()
)

repeated_stop_names.columns = ["stop_name", "n_rows"]

repeated_stop_names.head(20)

,stop_name,n_rows
0,Fridhem,25
1,Berga,22
2,Hagen,18
3,Heden,15
4,Mon,14
5,Haga,14
6,Marieberg,14
7,Åsen,14
8,Nygård,14
9,Berg,13


## Observation

The Swedish GTFS stop table is mostly detailed and clean, but stop names are not unique.

- There are 48,778 stop rows
- There are 44,421 unique stop names
- 4,357 rows share a stop name with another row
- Only 34 coordinate pairs are repeated
- This means that repeated names usually refer to different physical locations, not the exact same coordinate point


## Inspect GTFS calendar coverage

After checking stops and routes, I now inspect the GTFS calendar data.

For the calendar-level MVD, both `calendar.txt` and `calendar_dates.txt` are important.  
The active service dates later need to be rebuilt from:

- weekly rules in `calendar.txt`
- exceptions in `calendar_dates.txt`

First, I check whether all service IDs used in `trips.txt` are covered by the calendar files.

In [19]:
# Check GTFS calendar service_id coverage

calendar_service_ids = set(calendar["service_id"].astype(str))
calendar_dates_service_ids = set(calendar_dates["service_id"].astype(str))
trip_service_ids = set(trips["service_id"].astype(str))

all_calendar_service_ids = calendar_service_ids | calendar_dates_service_ids

gtfs_calendar_coverage = pd.DataFrame({
    "check": [
        "service IDs in calendar.txt",
        "service IDs in calendar_dates.txt",
        "service IDs used in trips.txt",
        "trip service IDs found in calendar or calendar_dates",
        "trip service IDs missing from calendar files",
        "service IDs only in calendar_dates.txt"
    ],
    "value": [
        len(calendar_service_ids),
        len(calendar_dates_service_ids),
        len(trip_service_ids),
        len(trip_service_ids & all_calendar_service_ids),
        len(trip_service_ids - all_calendar_service_ids),
        len(calendar_dates_service_ids - calendar_service_ids)
    ]
})

gtfs_calendar_coverage

,check,value
0,service IDs in calendar.txt,6893
1,service IDs in calendar_dates.txt,6892
2,service IDs used in trips.txt,6892
3,trip service IDs found in calendar or calendar...,6892
4,trip service IDs missing from calendar files,0
5,service IDs only in calendar_dates.txt,0


## Observation

The Swedish GTFS calendar coverage looks complete.

- `calendar.txt` contains 6,893 service IDs
- `calendar_dates.txt` contains 6,892 service IDs
- `trips.txt` uses 6,892 service IDs
- All service IDs used in trips are found in the calendar files
- No trip service IDs are missing
- There are no service IDs that exist only in `calendar_dates.txt`


## Inspect GTFS calendar date ranges and exceptions

The previous check showed that all service IDs used in trips are covered by the calendar files.

Before rebuilding active service dates, I inspect the validity period and exception types.

This helps to understand:
- the date range covered by `calendar.txt`
- the date range covered by `calendar_dates.txt`
- whether exceptions are additions, removals, or both

In [20]:
# Convert date columns to datetime

calendar_check = calendar.copy()
calendar_dates_check = calendar_dates.copy()

calendar_check["start_date"] = pd.to_datetime(calendar_check["start_date"].astype(str), format="%Y%m%d")
calendar_check["end_date"] = pd.to_datetime(calendar_check["end_date"].astype(str), format="%Y%m%d")

calendar_dates_check["date"] = pd.to_datetime(calendar_dates_check["date"].astype(str), format="%Y%m%d")

gtfs_calendar_date_summary = pd.DataFrame({
    "check": [
        "calendar start date min",
        "calendar end date max",
        "calendar_dates date min",
        "calendar_dates date max",
        "exception_type values"
    ],
    "value": [
        calendar_check["start_date"].min(),
        calendar_check["end_date"].max(),
        calendar_dates_check["date"].min(),
        calendar_dates_check["date"].max(),
        sorted(calendar_dates_check["exception_type"].dropna().unique().tolist())
    ]
})

gtfs_calendar_date_summary

,check,value
0,calendar start date min,2026-05-01 00:00:00
1,calendar end date max,2027-04-30 00:00:00
2,calendar_dates date min,2026-05-01 00:00:00
3,calendar_dates date max,2027-04-30 00:00:00
4,exception_type values,[1]


## Observation

The Swedish GTFS calendar has a clear validity window.

- The calendar starts on 2026-05-01 and ends on 2027-04-30
- `calendar_dates.txt` covers the same date range
- Only exception type `1` appears, meaning added service dates (so no removal exceptions in this feed)

This makes the GTFS calendar structure simpler than Norway, because the exceptions only add active dates.

## Inspect GTFS weekly calendar patterns

Before rebuilding active service dates, I inspect the weekday rules in `calendar.txt`.

This helps to see whether services are mainly defined by regular weekly patterns, or whether many services have no active weekday flags and are only activated through `calendar_dates.txt`.

This is important for Sweden because `calendar_dates.txt` only contains added service dates.

In [21]:
# Inspect weekly activity patterns in calendar.txt

weekday_cols = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]

calendar_check["n_weekdays_active"] = calendar_check[weekday_cols].sum(axis=1)

weekly_pattern_summary = pd.DataFrame({
    "check": [
        "calendar rows",
        "rows with at least one active weekday",
        "rows with no active weekdays",
        "unique weekly patterns"
    ],
    "value": [
        len(calendar_check),
        (calendar_check["n_weekdays_active"] > 0).sum(),
        (calendar_check["n_weekdays_active"] == 0).sum(),
        calendar_check[weekday_cols].drop_duplicates().shape[0]
    ]
})

weekly_pattern_summary

,check,value
0,calendar rows,6893
1,rows with at least one active weekday,0
2,rows with no active weekdays,6893
3,unique weekly patterns,1


## Observation

The Swedish GTFS calendar does not use regular weekly patterns.

- There are 6,893 rows in `calendar.txt` and none of those rows have an active weekday flag
- All rows have Monday to Sunday set to `0`

This means that the actual active service dates are defined through `calendar_dates.txt`, not through regular weekly rules.  

## Inspect GTFS active dates from calendar_dates

The weekly flags in `calendar.txt` are all inactive.  
This means that Swedish GTFS service validity is mainly defined through `calendar_dates.txt`.

Since `calendar_dates.txt` only contains exception type `1`, these rows represent added active service dates.

I now count how many active dates each `service_id` has.

In [22]:
# Count active dates per service_id from calendar_dates.txt

gtfs_active_dates_summary = (
    calendar_dates_check
    .groupby("service_id")["date"]
    .agg(
        n_active_dates="nunique",
        first_active_date="min",
        last_active_date="max"
    )
    .reset_index()
)

gtfs_active_dates_summary_check = pd.DataFrame({
    "check": [
        "service IDs with active dates",
        "minimum active dates per service_id",
        "maximum active dates per service_id",
        "average active dates per service_id"
    ],
    "value": [
        gtfs_active_dates_summary["service_id"].nunique(),
        gtfs_active_dates_summary["n_active_dates"].min(),
        gtfs_active_dates_summary["n_active_dates"].max(),
        round(gtfs_active_dates_summary["n_active_dates"].mean(), 2)
    ]
})

gtfs_active_dates_summary_check

,check,value
0,service IDs with active dates,6892.00
1,minimum active dates per service_id,1.00
2,maximum active dates per service_id,365.00
3,average active dates per service_id,33.63


## Comment

- The Swedish GTFS feed contains **6,892 unique service IDs** with at least one active date
- The minimum of **1 active date** per service ID suggests some services run on a single
  specific day only
- The maximum of **365 active dates** means some services run every day of the year
- The average of **33.63 active dates** per service ID indicates that most services run
  on a limited number of days

## Inspect distribution of GTFS active-date counts

The previous step showed that Swedish GTFS service IDs can have very different validity lengths.

Some service IDs are active for only one day, while others are active for a full year.

I now inspect the distribution of active-date counts.  


In [23]:
# Distribution of number of active dates per service_id

active_date_count_distribution = (
    gtfs_active_dates_summary["n_active_dates"]
    .value_counts()
    .reset_index()
)

active_date_count_distribution.columns = ["n_active_dates", "n_service_ids"]

active_date_count_distribution = active_date_count_distribution.sort_values(
    "n_active_dates"
)

active_date_count_distribution.head(20)

,n_active_dates,n_service_ids
5,1,213
0,2,528
1,3,435
2,4,372
3,5,309
4,6,276
6,7,201
7,8,184
9,9,143
10,10,136


In [24]:
# Service IDs with the most active dates

gtfs_active_dates_summary.sort_values(
    "n_active_dates",
    ascending=False
).head(20)

,service_id,n_active_dates,first_active_date,last_active_date
0,0,365,2026-05-01,2027-04-30
133,174,226,2026-05-01,2026-12-12
137,180,225,2026-05-02,2026-12-12
1880,2567,221,2026-05-06,2026-12-12
2487,3325,220,2026-05-06,2026-12-12
5110,6661,220,2026-05-06,2026-12-12
5109,6660,220,2026-05-06,2026-12-12
5345,6955,220,2026-05-06,2026-12-12
4737,6214,220,2026-05-06,2026-12-12
5392,7011,220,2026-05-06,2026-12-12


## Observation

The Swedish GTFS calendar contains both short and long service patterns.

- Many service IDs are active for only a small number of days
- 213 service IDs are active for only 1 day
- 528 service IDs are active for 2 days
- Some service IDs are active for much longer periods
- One service ID is active for the full calendar period of 365 days
- Many long-running services end on 2026-12-12, which may indicate a timetable change period

This shows that the Swedish GTFS calendar is quite detailed and is mainly built from explicit active dates in `calendar_dates.txt`.

## Create GTFS calendar summary for comparison

For the later GTFS–NeTEx calendar comparison, I create one summary table per `service_id`.

This table keeps:

- the service ID
- number of active days
- first active date
- last active date


In [25]:
# Create GTFS calendar summary for later comparison

gtfs_calendar_summary = gtfs_active_dates_summary.copy()

gtfs_calendar_summary = gtfs_calendar_summary.rename(columns={
    "n_active_dates": "n_active_days"
})

gtfs_calendar_summary["service_id"] = gtfs_calendar_summary["service_id"].astype("string")

gtfs_calendar_summary_check = pd.DataFrame({
    "check": [
        "calendar summary rows",
        "unique service_id values",
        "missing service_id",
        "missing n_active_days",
        "missing first_active_date",
        "missing last_active_date"
    ],
    "value": [
        len(gtfs_calendar_summary),
        gtfs_calendar_summary["service_id"].nunique(),
        gtfs_calendar_summary["service_id"].isna().sum(),
        gtfs_calendar_summary["n_active_days"].isna().sum(),
        gtfs_calendar_summary["first_active_date"].isna().sum(),
        gtfs_calendar_summary["last_active_date"].isna().sum()
    ]
})

gtfs_calendar_summary_check

,check,value
0,calendar summary rows,6892
1,unique service_id values,6892
2,missing service_id,0
3,missing n_active_days,0
4,missing first_active_date,0
5,missing last_active_date,0


In [26]:
gtfs_calendar_summary.head(10)

,service_id,n_active_days,first_active_date,last_active_date
0,0,365,2026-05-01,2027-04-30
1,1,5,2026-05-09,2026-06-13
2,3,40,2026-06-22,2026-08-14
3,5,9,2026-06-20,2026-08-09
4,7,32,2026-05-05,2026-06-18
5,8,45,2026-05-05,2026-06-18
6,9,13,2026-05-09,2026-06-14
7,11,18,2026-06-19,2026-08-15
8,12,58,2026-06-19,2026-08-15
9,13,8,2026-05-10,2026-06-14


## Observation

The GTFS calendar summary was created successfully.

- The summary contains 6,892 service IDs
- Each service ID appears only once and there are no missing values in the key fields
- The table keeps the number of active days, the first active date, and the last active date


## Check GTFS trips as bridge table

Before finishing the GTFS exploration, I check whether `trips.txt` connects correctly to the route and calendar tables.

This is useful because `trips.txt` links the main GTFS concepts:

- `trips.route_id` &rarr; `routes.txt`
- `trips.service_id` &rarr; calendar validity

So this step checks whether the route and calendar information can be connected later if needed.

In [27]:
# Check whether trips connect correctly to routes and calendar

trip_route_ids = set(trips["route_id"].astype(str))
route_ids = set(routes_check["route_id"].astype(str))

trip_service_ids = set(trips["service_id"].astype(str))
calendar_summary_service_ids = set(gtfs_calendar_summary["service_id"].astype(str))

gtfs_trip_bridge_check = pd.DataFrame({
    "check": [
        "trip rows",
        "unique route_id values in trips",
        "trip route_ids found in routes",
        "trip route_ids missing from routes",
        "unique service_id values in trips",
        "trip service_ids found in calendar summary",
        "trip service_ids missing from calendar summary"
    ],
    "value": [
        len(trips),
        len(trip_route_ids),
        len(trip_route_ids & route_ids),
        len(trip_route_ids - route_ids),
        len(trip_service_ids),
        len(trip_service_ids & calendar_summary_service_ids),
        len(trip_service_ids - calendar_summary_service_ids)
    ]
})

gtfs_trip_bridge_check

,check,value
0,trip rows,523669
1,unique route_id values in trips,9153
2,trip route_ids found in routes,9153
3,trip route_ids missing from routes,0
4,unique service_id values in trips,6892
5,trip service_ids found in calendar summary,6891
6,trip service_ids missing from calendar summary,1


## Observation

The GTFS trip connections are almost complete.

- `trips.txt` contains 523,669 trip rows
- All 9,153 route IDs used in trips are found in `routes.txt`
- This means the connection `trips.route_id` &rarr; `routes.txt` works completely
- For the calendar side, 6,891 out of 6,892 service IDs are found in the calendar summary
- Only 1 service ID is missing from the calendar summary

This is only a small edge case.  
It probably means that this service ID exists in the GTFS files, but has no active dates in the final active-date summary.

## Inspect missing service_id from calendar summary
     

In [28]:
# Find service_ids used in trips but missing from the calendar summary

missing_calendar_service_ids = trip_service_ids - calendar_summary_service_ids

missing_calendar_service_ids

{'3413'}

In [29]:
# Inspect the missing service_id in calendar, calendar_dates, and trips

missing_service_id = list(missing_calendar_service_ids)[0]

print("Missing service_id:", missing_service_id)

print("\nIn calendar.txt:")
display(calendar[calendar["service_id"].astype(str) == missing_service_id])

print("\nIn calendar_dates.txt:")
display(calendar_dates[calendar_dates["service_id"].astype(str) == missing_service_id])

print("\nIn trips.txt:")
display(trips[trips["service_id"].astype(str) == missing_service_id].head(20))

Missing service_id: 3413

In calendar.txt:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
2557,3413,0,0,0,0,0,0,0,20260501,20270430



In calendar_dates.txt:


,service_id,date,exception_type



In trips.txt:


,route_id,service_id,trip_id,trip_headsign,trip_short_name
120709,1300000000005,3413,23000000010099,Halmstad Centralstation,20189
365203,1300080400045,3413,23000804010262,Halmstad Centralstation,1146


## Observation

The missing calendar service ID is `3413`.

- It exists in `calendar.txt` and it is used in `trips.txt`
- It has no active weekdays in `calendar.txt` and no added dates in `calendar_dates.txt`

This means that service ID `3413` is used by trips, but it has no active service dates.  
Therefore, it was not included in the final GTFS calendar summary.

This is a small data inconsistency, but it does not affect the main GTFS exploration.

## Summary of GTFS exploration


At the stop level, the feed is different from Norway.  
There is no `parent_station` column, and all `location_type` values are missing.  
Therefore, the GTFS stop hierarchy cannot be separated into stations and platforms. For Sweden, all GTFS stop rows will be treated as stop-level records.

At the route level, `route_short_name` is useful but not complete.  
Many routes without `route_short_name` still have a meaningful `route_long_name.  
Therefore, Sweden uses a combined GTFS public label:

`route_short_name` &rarr; if missing, use `route_long_name`

The route table is multimodal and includes bus, rail, express bus, tram, water transport, metro / urban rail, and taxi-like services.

At the calendar level, Sweden does not use regular weekly patterns in `calendar.txt`.  
All weekday flags are set to 0. The active service dates are mainly defined through `calendar_dates.txt`, which only contains added dates.

The GTFS calendar summary was created successfully for 6,892 service IDs.  
One service ID is used in trips but has no active dates, so it was excluded from the active-date summary.

Overall, the Swedish GTFS feed is usable for the comparison, but it needs some adaptations compared with Norway:
- no station/platform hierarchy on the GTFS stop side
- combined public label needed for routes
- calendar validity mainly based on explicit dates

# Part 2: NeTEx Exploration

In [30]:
# Set NeTEx path
netex_zip_path = data_dir / "NeTEx Sweden Static data.zip"
print("NeTEx path:")
print(netex_zip_path)
print("\nNeTEx file exists:")
print(netex_zip_path.exists())

NeTEx path:
data/NeTEx Sweden Static data.zip

NeTEx file exists:
True


In [31]:
# Inspect NeTEx ZIP contents
with zipfile.ZipFile(netex_zip_path, "r") as z:
    netex_files = pd.DataFrame([
        {
            "name": info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(netex_files.sort_values("size_mb", ascending=False).head(30))

print("Number of files in NeTEx ZIP:", len(netex_files))

,name,size_mb
4167,_shared_data.xml,681.94
4106,_stops.xml,192.21
1496,line_1_0_9011001000000000.xml,120.62
3211,line_14_5006_9011014500600000.xml,106.64
1889,line_14_5011_9011014501100000.xml,81.45
888,line_14_5007_9011014500700000.xml,72.19
4342,line_14_5005_9011014500500000.xml,71.55
6669,line_14_5003_9011014500300000.xml,66.25
1819,line_14_5001_9011014500100000.xml,65.29
4964,line_790_0_9011790000000000.xml,52.56


Number of files in NeTEx ZIP: 6893


## Observation

The Swedish NeTEx ZIP contains 6,893 XML files.

- The feed is split into many XML files
- There is one large `_shared_data.xml` file, about 682 MB
- There is also a large `_stops.xml` file, about 192 MB
- Many other files are line-specific XML files, for example files starting with `line_...`

This structure is similar to Norway in the sense that stops, shared data, and line files are separated.  

## Inspect NeTEx file structure

Before extracting data, I first check which important elements appear in the main file types.

This is important because the Swedish NeTEx feed is large and split into many XML files.

I want to check where the main comparison objects are stored:

- `StopPlace` and `Quay` &rarr; stop / station level
- `Line`, `Route`, and `ServiceJourney` &rarr; line / route level
- `DayType`, `DayTypeAssignment`, and operating periods &rarr; calendar / validity level



In [32]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    netex_file_names = sorted(z.namelist())

print(f"Total files: {len(netex_file_names)}")
print("\nUnique file name patterns:")
patterns = sorted(set(re.sub(r'\d+', 'N', f) for f in netex_file_names))
for p in patterns[:30]:
    print(p)

print("\nFirst 10 files:")
for f in netex_file_names[:10]:
    print(f)

Total files: 6893

Unique file name patterns:
_shared_data.xml
_stops.xml
line_N_N_N.xml

First 10 files:
_shared_data.xml
_stops.xml
line_100_0_9011100000000000.xml
line_100_100_9011100010000000.xml
line_100_101_9011100010100000.xml
line_100_102_9011100010200000.xml
line_100_103_9011100010300000.xml
line_100_106_9011100010600000.xml
line_100_108_9011100010800000.xml
line_100_109_9011100010900000.xml


In [33]:
# Select main NeTEx files for structure inspection
stops_file   = "_stops.xml"
shared_file  = "_shared_data.xml"
line_files   = [f for f in netex_file_names if f.startswith("line_")]
sample_line_file = line_files[0] if line_files else None

print(f"Stops file:        {stops_file}")
print(f"Shared data file:  {shared_file}")
print(f"Number of line files: {len(line_files)}")
print(f"Sample line file:  {sample_line_file}")

Stops file:        _stops.xml
Shared data file:  _shared_data.xml
Number of line files: 6891
Sample line file:  line_100_0_9011100000000000.xml


In [34]:
# XML parsing helpers for the NeTEx exploration
# (same helpers used in the Norway notebook, reused here for Sweden)

def local_name(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def check_elements_in_zip_xml(zip_path, member_name, target_elements, max_elements=300000):
    found = {e: False for e in target_elements}
    n_checked = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                n_checked += 1

                if tag in found:
                    found[tag] = True

                elem.clear()

                if all(found.values()) or n_checked >= max_elements:
                    break

    return found, n_checked

In [35]:
# Check where important NeTEx elements appear across the three file types
checks = [
    {
        "file_role": "stops file",
        "file_name": stops_file,
        "target_elements": ["StopPlace", "Quay"]
    },
    {
        "file_role": "sample line file",
        "file_name": sample_line_file,
        "target_elements": ["Line", "Route", "ServiceJourney"]
    },
    {
        "file_role": "shared data file",
        "file_name": shared_file,
        "target_elements": [
            "DayType",
            "DayTypeAssignment",
            "OperatingPeriod",
            "UicOperatingPeriod",
            "FromDate",
            "ToDate",
            "ValidDayBits"
        ]
    }
]

structure_checks = []
for check in checks:
    print(f"  Checking {check['file_name']}...", end="\r")
    found, n_checked = check_elements_in_zip_xml(
        netex_zip_path,
        check["file_name"],
        check["target_elements"]
    )
    row = {
        "file_role":        check["file_role"],
        "file_name":        check["file_name"],
        "elements_checked": n_checked
    }
    row.update(found)
    structure_checks.append(row)

print("Done.                          ")
netex_structure_check = pd.DataFrame(structure_checks)
display(netex_structure_check)

Done.                          0000000.xml...


,file_role,file_name,elements_checked,StopPlace,Quay,Line,Route,ServiceJourney,DayType,DayTypeAssignment,OperatingPeriod,UicOperatingPeriod,FromDate,ToDate,ValidDayBits
0,stops file,_stops.xml,77384,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sample line file,line_100_0_9011100000000000.xml,5740,NaN,NaN,True,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,shared data file,_shared_data.xml,300000,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True,False


## Observation


The file structure is quite clear:

- one `_stops.xml` file
- one `_shared_data.xml` file
- 6,891 line-specific XML files

The structure check shows that:

- `_stops.xml` contains both `StopPlace` and `Quay`
- the sampled line file contains `Line`, `Route`, and `ServiceJourney`
- `_shared_data.xml` contains date information such as `FromDate` and `ToDate`

This confirms that the Swedish NeTEx feed is split by function:

- `_stops.xml` &rarr; stop / station information
- line files &rarr; line, route, and journey information
- `_shared_data.xml` &rarr; shared calendar / validity information


## Inspect sample StopPlace and Quay records

Before extracting all stops, I inspect a small sample of both `StopPlace` and `Quay`
elements from `_stops.xml`. This helps to understand which NeTEx level is closer to
the flat Swedish GTFS stop table, and to confirm that names and coordinates are
available in the data.

In [36]:
def extract_stopplace_quay_sample(zip_path, xml_file, n_sample=10):
    samples = []
    counts  = {"StopPlace": 0, "Quay": 0}
    current = None
    depth   = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)

                if event == "start" and tag in ["StopPlace", "Quay"]:
                    depth += 1
                    if depth == 1:
                        current = {
                            "element_type": tag,
                            "id":           elem.attrib.get("id"),
                            "name":         None,
                            "latitude":     None,
                            "longitude":    None
                        }

                elif event == "end" and tag in ["StopPlace", "Quay"]:
                    if depth == 1 and current is not None:
                        if counts[tag] < n_sample:
                            samples.append(current)
                            counts[tag] += 1
                        current = None
                    depth -= 1
                    elem.clear()

                elif event == "end" and current is not None and depth > 0:
                    text = elem.text.strip() if elem.text else None
                    if tag == "Name" and current["name"] is None:
                        current["name"] = text
                    elif tag == "Latitude" and current["latitude"] is None:
                        current["latitude"] = text
                    elif tag == "Longitude" and current["longitude"] is None:
                        current["longitude"] = text
                    elem.clear()

                elif event == "end":
                    elem.clear()

                if counts["StopPlace"] >= n_sample and counts["Quay"] >= n_sample:
                    break

    print(f"StopPlaces sampled: {counts['StopPlace']}")
    print(f"Quays sampled:      {counts['Quay']}")
    return pd.DataFrame(samples)

netex_stop_sample = extract_stopplace_quay_sample(
    netex_zip_path,
    "_stops.xml",
    n_sample=10
)
display(netex_stop_sample)

StopPlaces sampled: 10
Quays sampled:      0


,element_type,id,name,latitude,longitude
0,StopPlace,SE:030:StopPlace:9021050002025000,Tyresta,59.667474,17.247889
1,StopPlace,SE:030:StopPlace:9021050002026000,Getingbo,60.184895,17.472558
2,StopPlace,SE:030:StopPlace:9021050002027000,Rävsta,59.695940,17.480298
3,StopPlace,SE:030:StopPlace:9021050002028000,Jerusalem,59.726317,17.364606
4,StopPlace,SE:030:StopPlace:9021050002029000,Kongo,59.759930,17.372809
5,StopPlace,SE:030:StopPlace:9021050002030000,Ölsta,59.753592,17.392244
6,StopPlace,SE:030:StopPlace:9021050002031000,Eningböle gård,59.750779,17.339575
7,StopPlace,SE:030:StopPlace:9021050002032000,Varstabacken,59.713414,17.283274
8,StopPlace,SE:030:StopPlace:9021050002033000,Uppeby,59.554900,17.361277
9,StopPlace,SE:030:StopPlace:9021050002034000,Lerbäck,59.514838,17.292761


## Comment

Names and coordinates are available for all sampled
`StopPlace` elements. 

No `Quay` elements were sampled. This suggests that Quays are nested inside StopPlace
elements rather than appearing as separate top-level elements. The next step checks
this before deciding which NeTEx level to use for the stop comparison.

In [37]:
# Check whether Quay appears as a top-level or nested element
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open("_stops.xml") as f:
        depth = 0
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            if event == "start":
                depth += 1
            if event == "start" and tag == "Quay":
                print(f"Quay found at depth {depth}")
                elem.clear()
                break
            if event == "end":
                depth -= 1
                elem.clear()

Quay found at depth 7


Quays are deeply nested inside StopPlace elements (at depth 7), not top-level elements. This means the Swedish NeTEx stop structure looks something like:

StopPlace
  
     └── quays
        
            └── Quay  ← depth 7

## Extract all NeTEx StopPlaces

The sample showed that `StopPlace` elements contain names and coordinates at the
top level, while `Quay` elements are nested inside at depth 7. Since the Swedish
GTFS stop table is flat with no hierarchy, `StopPlace` is the closest NeTEx
equivalent and will be used for the stop-level comparison.

All `StopPlace` elements are now extracted from `_stops.xml` with their ID, name,
and coordinates.

In [38]:
# Extract all StopPlace elements from _stops.xml
def extract_all_stopplaces(zip_path, member_name):
    rows = []
    current = None
    depth = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)

                if event == "start" and tag == "StopPlace":
                    depth += 1
                    if depth == 1:
                        current = {
                            "netex_stopplace_id":   elem.attrib.get("id"),
                            "netex_stopplace_name": None,
                            "netex_lat":            None,
                            "netex_lon":            None
                        }

                elif event == "end" and tag == "StopPlace":
                    if depth == 1 and current is not None:
                        rows.append(current)
                        current = None
                    depth -= 1
                    elem.clear()

                elif event == "end" and depth > 0 and current is not None:
                    text = elem.text.strip() if elem.text else None
                    if tag == "Name" and current["netex_stopplace_name"] is None:
                        current["netex_stopplace_name"] = text
                    elif tag == "Latitude" and current["netex_lat"] is None:
                        current["netex_lat"] = text
                    elif tag == "Longitude" and current["netex_lon"] is None:
                        current["netex_lon"] = text
                    elem.clear()

                elif event == "end":
                    elem.clear()

    print(f"\nDone — {len(rows)} StopPlaces extracted.")
    return pd.DataFrame(rows)

netex_stopplaces = extract_all_stopplaces(netex_zip_path, "_stops.xml")

# Convert coordinates to numeric
netex_stopplaces["netex_lat"] = pd.to_numeric(netex_stopplaces["netex_lat"], errors="coerce")
netex_stopplaces["netex_lon"] = pd.to_numeric(netex_stopplaces["netex_lon"], errors="coerce")

display(netex_stopplaces.head(10))


Done — 71650 StopPlaces extracted.


,netex_stopplace_id,netex_stopplace_name,netex_lat,netex_lon
0,SE:030:StopPlace:9021050002025000,Tyresta,59.667474,17.247889
1,SE:030:StopPlace:9021050002026000,Getingbo,60.184895,17.472558
2,SE:030:StopPlace:9021050002027000,Rävsta,59.695940,17.480298
3,SE:030:StopPlace:9021050002028000,Jerusalem,59.726317,17.364606
4,SE:030:StopPlace:9021050002029000,Kongo,59.759930,17.372809
5,SE:030:StopPlace:9021050002030000,Ölsta,59.753592,17.392244
6,SE:030:StopPlace:9021050002031000,Eningböle gård,59.750779,17.339575
7,SE:030:StopPlace:9021050002032000,Varstabacken,59.713414,17.283274
8,SE:030:StopPlace:9021050002033000,Uppeby,59.554900,17.361277
9,SE:030:StopPlace:9021050002034000,Lerbäck,59.514838,17.292761


## Comment

71,650 `StopPlace` elements were successfully extracted from `_stops.xml`.
All sampled rows have a name and coordinates, confirming that the extraction
works correctly for Sweden.


# Data quality check on the extracted StopPlaces

In [39]:
# NeTEx StopPlace data quality check
netex_stopplace_quality = pd.DataFrame({
    "metric": [
        "total StopPlace rows",
        "rows with name",
        "rows missing name",
        "rows with coordinates",
        "rows missing coordinates",
        "unique StopPlace IDs",
        "duplicate StopPlace IDs"
    ],
    "value": [
        len(netex_stopplaces),
        netex_stopplaces["netex_stopplace_name"].notna().sum(),
        netex_stopplaces["netex_stopplace_name"].isna().sum(),
        netex_stopplaces[["netex_lat", "netex_lon"]].notna().all(axis=1).sum(),
        netex_stopplaces[["netex_lat", "netex_lon"]].isna().any(axis=1).sum(),
        netex_stopplaces["netex_stopplace_id"].nunique(),
        netex_stopplaces["netex_stopplace_id"].duplicated().sum()
    ]
})
display(netex_stopplace_quality)

,metric,value
0,total StopPlace rows,71650
1,rows with name,71650
2,rows missing name,0
3,rows with coordinates,71650
4,rows missing coordinates,0
5,unique StopPlace IDs,71650
6,duplicate StopPlace IDs,0


## Comment

The NeTEx StopPlace table is complete and clean:

- All 71,650 StopPlace rows have a name, coordinates, and a unique ID
- There are no missing values and no duplicate IDs


## Inspect NeTEx line structure

Before extracting all lines, I check whether the Swedish NeTEx line files follow
the same structure as Norway. 

In [40]:
# Check structure of a sample line file
found, n_checked = check_elements_in_zip_xml(
    netex_zip_path,
    sample_line_file,
    ["Line", "Route", "ServiceJourney", "PublicCode", "TransportMode"]
)
print(f"Elements checked: {n_checked}")
for element, present in found.items():
    print(f"  {element}: {present}")

Elements checked: 5740
  Line: True
  Route: True
  ServiceJourney: True
  PublicCode: True
  TransportMode: True


## Comment

All expected elements are present in the Swedish NeTEx line files:
`Line`, `Route`, `ServiceJourney`, `PublicCode`, and `TransportMode`.
The structure is identical to Norway.

## Extract all NeTEx Lines

The structure check confirmed that the Swedish NeTEx line files follow the same
structure as Norway. So I can reuse the same extraction function to obtain
all `Line` elements from the 6,891 line-specific XML files, including the public
code, transport mode, and file-level route and service journey counts.

In [41]:
# Extract all NeTEx Line elements from line-specific XML files
def extract_all_lines(zip_path, file_list):
    rows = []
    total = len(file_list)

    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{total}] {member_name}", end="\r")
            file_lines = []
            route_count = 0
            service_journey_count = 0
            current_line = None
            line_depth = 0

            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "Line":
                        line_depth += 1
                        if line_depth == 1:
                            current_line = {
                                "file":                 member_name,
                                "netex_line_id":        elem.attrib.get("id"),
                                "netex_line_name":      None,
                                "netex_short_name":     None,
                                "netex_public_code":    None,
                                "netex_transport_mode": None
                            }

                    elif event == "end" and tag == "Line":
                        if line_depth == 1 and current_line is not None:
                            file_lines.append(current_line)
                            current_line = None
                        line_depth -= 1
                        elem.clear()

                    elif event == "end" and tag != "Line" and current_line is not None and line_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "Name" and current_line["netex_line_name"] is None:
                            current_line["netex_line_name"] = text
                        elif tag == "ShortName" and current_line["netex_short_name"] is None:
                            current_line["netex_short_name"] = text
                        elif tag == "PublicCode" and current_line["netex_public_code"] is None:
                            current_line["netex_public_code"] = text
                        elif tag == "TransportMode" and current_line["netex_transport_mode"] is None:
                            current_line["netex_transport_mode"] = text
                        elem.clear()

                    elif event == "end":
                        if tag == "Route":
                            route_count += 1
                        elif tag == "ServiceJourney":
                            service_journey_count += 1
                        elem.clear()

            for line in file_lines:
                line["route_count"] = route_count
                line["service_journey_count"] = service_journey_count
                rows.append(line)

    print(f"\nDone — {len(rows)} lines extracted from {total} files.")
    return pd.DataFrame(rows)

netex_lines = extract_all_lines(netex_zip_path, line_files)
display(netex_lines.head(20))

  [6891/6891] line_9_6_9011009000600000.xmllllll
Done — 6891 lines extracted from 6891 files.


,file,netex_line_id,netex_line_name,netex_short_name,netex_public_code,netex_transport_mode,route_count,service_journey_count
0,line_100_0_9011100000000000.xml,SE:030:Line:9011100000000000,SJ,None,74,rail,81,81
1,line_100_100_9011100010000000.xml,SE:030:Line:9011100010000000,100,None,100,rail,1,7
2,line_100_101_9011100010100000.xml,SE:030:Line:9011100010100000,101,None,101,rail,1,7
3,line_100_102_9011100010200000.xml,SE:030:Line:9011100010200000,102,None,102,rail,1,7
4,line_100_103_9011100010300000.xml,SE:030:Line:9011100010300000,103,None,103,rail,1,6
5,line_100_106_9011100010600000.xml,SE:030:Line:9011100010600000,106,None,106,rail,1,4
6,line_100_108_9011100010800000.xml,SE:030:Line:9011100010800000,108,None,108,rail,1,12
7,line_100_109_9011100010900000.xml,SE:030:Line:9011100010900000,109,None,109,rail,1,4
8,line_100_10_9011100001000000.xml,SE:030:Line:9011100001000000,10,None,10,rail,7,43
9,line_100_110_9011100011000000.xml,SE:030:Line:9011100011000000,110,None,110,rail,2,14


## Comment

6,891 lines were successfully extracted from all line files. A few observations:

- `netex_short_name` is `None` for all sampled rows. Sweden appears to not use
  this field, which is fine since `netex_public_code` is always populated
    
- The public codes are numeric (100, 101, 102...) and match the `netex_line_name`,
  suggesting they are the same value stored in both fields
    
- All sampled lines have `rail` as transport mode. This is because the first files
  are from operator `100` which is SJ (Swedish national rail). Other transport modes
  will appear in later files



## Prepare NeTEx line-level dataset

The raw extraction produced one row per line file. Before comparing with GTFS,
the NeTEx line table is deduplicated by `public_code` so that each unique line
label appears only once. Lines without a public code are excluded, as they cannot
be matched to GTFS routes.

In [42]:
# Prepare NeTEx line-level dataset
netex_lines_with_code = netex_lines[netex_lines["netex_public_code"].notna()].copy()

netex_line_summary = pd.DataFrame({
    "metric": [
        "Line rows",
        "Line rows with public_code",
        "Line rows without public_code",
        "unique public_code values",
        "duplicate public_code rows"
    ],
    "value": [
        len(netex_lines),
        len(netex_lines_with_code),
        netex_lines["netex_public_code"].isna().sum(),
        netex_lines_with_code["netex_public_code"].nunique(),
        netex_lines_with_code["netex_public_code"].duplicated().sum()
    ]
})
display(netex_line_summary)

netex_lines_dedup = (
    netex_lines_with_code
    .groupby("netex_public_code", as_index=False)
    .agg(
        number_of_line_rows=("netex_line_id", "count"),
        transport_modes=("netex_transport_mode", lambda x: ", ".join(sorted(x.dropna().unique().tolist()))),
        example_line_name=("netex_line_name", "first"),
        route_count_total=("route_count", "sum"),
        service_journey_count_total=("service_journey_count", "sum")
    )
    .rename(columns={"netex_public_code": "netex_line_label"})
    .sort_values(
        "netex_line_label",
        key=lambda s: pd.to_numeric(s, errors="coerce").fillna(float("inf"))
    )
    .reset_index(drop=True)
)
display(netex_lines_dedup.head(20))

,metric,value
0,Line rows,6891
1,Line rows with public_code,6891
2,Line rows without public_code,0
3,unique public_code values,3431
4,duplicate public_code rows,3460


,netex_line_label,number_of_line_rows,transport_modes,example_line_name,route_count_total,service_journey_count_total
0,1,71,"bus, rail, taxi, tram, water",1,4873,30636
1,1.,1,bus,1.,98,98
2,2,66,"bus, rail, taxi, tram, water",2,4096,26894
3,2.,1,bus,2.,50,50
4,3.,1,bus,3.,36,36
5,3,52,"bus, tram, water",3,2268,22869
6,4.,1,bus,4.,92,92
7,4,41,"bus, taxi, tram, water",4,1965,20866
8,5,31,"bus, taxi, tram, water",5,1804,19974
9,5.,1,bus,5.,5,5


## Comment

All 6,891 NeTEx line rows have a public code, so no rows are excluded.
After deduplication, **3,431 unique line labels** remain, with 3,460 duplicate
rows, meaning many line labels appear across multiple files, likely because
the same line number is used by different regional operators.

Two interesting patterns stand out:

- Labels such as "1." and "2." appear alongside "1" and "2". These may be
  formatting variants or distinct regional line labels. In this notebook they
  are kept separate to avoid overmatching, but this can still contribute to
  apparent mismatches (see the line-level comparison below).
  
- Low-numbered labels like "1" and "2" cover up to 71 and 66 line rows
  respectively, with multiple transport modes (bus, rail, taxi, tram, water).
  This reflects Sweden's highly fragmented operator structure where the same
  line number is reused across many regional operators and modes.

## Inspect NeTEx calendar structure

Before extracting calendar data, I inspect which calendar elements are present
in `_shared_data.xml`. This determines how Sweden encodes service validity,
whether through `ValidDayBits` (as in Luxembourg and France), through
`OperatingPeriod` date ranges (as in Norway), or through a different approach.

In [43]:
# Inspect the Swedish NeTEx shared data file structure
def count_elements_in_file(zip_path, member_name, target_elements):
    counts = {e: 0 for e in target_elements}
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                if tag in counts:
                    counts[tag] += 1
                elem.clear()
    return counts

target_elements = [
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "UicOperatingPeriod",
    "FromDate",
    "ToDate",
    "ValidDayBits"
]

print(f"Counting elements in {shared_file}...")
counts = count_elements_in_file(netex_zip_path, shared_file, target_elements)

netex_calendar_structure = pd.DataFrame({
    "element": list(counts.keys()),
    "count":   list(counts.values())
})
display(netex_calendar_structure)

Counting elements in _shared_data.xml...


,element,count
0,DayType,7516
1,DayTypeAssignment,90141
2,OperatingPeriod,2595
3,UicOperatingPeriod,0
4,FromDate,198685
5,ToDate,12848
6,ValidDayBits,0


## Comment

Sweden uses the same calendar approach as Norway so `DayTypeAssignment` with
`OperatingPeriod` date ranges. No `ValidDayBits` are used, confirming that
the Norwegian extraction code can be reused directly.

A few things stand out:

- `FromDate` appears 198,685 times but `ToDate` only 12,848 times. This is
  a large difference and suggests that many assignments use a single `FromDate`
  as a direct date reference rather than a date range

- 7,516 `DayType` elements and 90,141 `DayTypeAssignment` elements. About
  12 assignments per DayType on average, similar to Norway's SKY operator
    
- 2,595 `OperatingPeriod` elements. Fewer than Norway, meaning most dates
  are likely assigned directly rather than through periods

## Extract NeTEx calendar data

Sweden uses the same calendar structure as Norway: `DayTypeAssignment` elements
pointing to either a specific date or an `OperatingPeriod` date range. The same
extraction functions from Norway are reused directly, with `_shared_data.xml` as
the only input file since Sweden has a single shared file instead of Norway's 67
operator-level files.

In [44]:
def extract_all_daytype_assignments(zip_path, file_list):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{len(file_list)}] {member_name}", end="\r")
            current_assignment = None
            assignment_depth = 0
            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "DayTypeAssignment":
                        assignment_depth += 1
                        if assignment_depth == 1:
                            current_assignment = {
                                "file":                 member_name,
                                "assignment_id":        elem.attrib.get("id"),
                                "daytype_ref":          None,
                                "operating_period_ref": None,
                                "date":                 None,
                                "is_available":         None
                            }

                    elif event == "end" and tag == "DayTypeAssignment":
                        if assignment_depth == 1 and current_assignment is not None:
                            rows.append(current_assignment)
                            current_assignment = None
                        assignment_depth -= 1
                        elem.clear()

                    elif event == "end" and current_assignment is not None and assignment_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "DayTypeRef" and current_assignment["daytype_ref"] is None:
                            current_assignment["daytype_ref"] = elem.attrib.get("ref")
                        elif tag == "OperatingPeriodRef" and current_assignment["operating_period_ref"] is None:
                            current_assignment["operating_period_ref"] = elem.attrib.get("ref")
                        elif tag == "Date" and current_assignment["date"] is None:
                            current_assignment["date"] = text
                        elif tag == "IsAvailable" and current_assignment["is_available"] is None:
                            current_assignment["is_available"] = text
                        elem.clear()

                    elif event == "end":
                        elem.clear()

    print(f"\nDone — {len(rows)} DayTypeAssignments extracted.")
    return pd.DataFrame(rows)

def extract_all_operating_periods(zip_path, file_list):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, member_name in enumerate(file_list, start=1):
            print(f"  [{i}/{len(file_list)}] {member_name}", end="\r")
            current_period = None
            period_depth = 0
            with z.open(member_name) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    if event == "start" and tag == "OperatingPeriod":
                        period_depth += 1
                        if period_depth == 1:
                            current_period = {
                                "file":                member_name,
                                "operating_period_id": elem.attrib.get("id"),
                                "from_date":           None,
                                "to_date":             None
                            }

                    elif event == "end" and tag == "OperatingPeriod":
                        if period_depth == 1 and current_period is not None:
                            rows.append(current_period)
                            current_period = None
                        period_depth -= 1
                        elem.clear()

                    elif event == "end" and current_period is not None and period_depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "FromDate" and current_period["from_date"] is None:
                            current_period["from_date"] = text
                        elif tag == "ToDate" and current_period["to_date"] is None:
                            current_period["to_date"] = text
                        elem.clear()

                    elif event == "end":
                        elem.clear()

    print(f"\nDone — {len(rows)} OperatingPeriods extracted.")
    return pd.DataFrame(rows)

In [45]:
netex_daytype_assignments = extract_all_daytype_assignments(
    netex_zip_path,
    [shared_file]
)
netex_daytype_assignments["date"] = pd.to_datetime(
    netex_daytype_assignments["date"], errors="coerce"
)
netex_daytype_assignments["is_available"] = netex_daytype_assignments["is_available"].map(
    {"true": True, "false": False}
)

netex_operating_periods = extract_all_operating_periods(
    netex_zip_path,
    [shared_file]
)
netex_operating_periods["from_date"] = pd.to_datetime(
    netex_operating_periods["from_date"], errors="coerce"
)
netex_operating_periods["to_date"] = pd.to_datetime(
    netex_operating_periods["to_date"], errors="coerce"
)

display(netex_daytype_assignments.head(10))
display(netex_operating_periods.head(10))

  [1/1] _shared_data.xml
Done — 90141 DayTypeAssignments extracted.
  [1/1] _shared_data.xml
Done — 2595 OperatingPeriods extracted.


,file,assignment_id,daytype_ref,operating_period_ref,date,is_available
0,_shared_data.xml,SE:030:DayTypeAssignment:utmge7pjhtmr530eqrpus...,SE:030:DayType:utmge7pjhtmr530eqrpus3h7lciasaet,SE:030:OperatingPeriod:260620260809,NaT,NaN
1,_shared_data.xml,SE:030:DayTypeAssignment:utmge7pjhtmr530eqrpus...,SE:030:DayType:utmge7pjhtmr530eqrpus3h7lciasaet,None,2026-06-20,NaN
2,_shared_data.xml,SE:030:DayTypeAssignment:u4hk10grb952gcnfvmka7...,SE:030:DayType:u4hk10grb952gcnfvmka7f0a3n6u2vv0,SE:030:OperatingPeriod:260706260807,NaT,NaN
3,_shared_data.xml,SE:030:DayTypeAssignment:vrk4q57ertmoi8opc3tnl...,SE:030:DayType:vrk4q57ertmoi8opc3tnl501ftkc1qsc,SE:030:OperatingPeriod:260530260530,NaT,NaN
4,_shared_data.xml,SE:030:DayTypeAssignment:2jett2pvjflsps8jvh15t...,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,SE:030:OperatingPeriod:260507260618,NaT,NaN
5,_shared_data.xml,SE:030:DayTypeAssignment:2jett2pvjflsps8jvh15t...,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,None,2026-05-14,NaN
6,_shared_data.xml,SE:030:DayTypeAssignment:2jett2pvjflsps8jvh15t...,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,None,2026-06-15,NaN
7,_shared_data.xml,SE:030:DayTypeAssignment:2jett2pvjflsps8jvh15t...,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,None,2026-06-16,NaN
8,_shared_data.xml,SE:030:DayTypeAssignment:2jett2pvjflsps8jvh15t...,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,None,2026-06-17,NaN
9,_shared_data.xml,SE:030:DayTypeAssignment:shtdvssbqnth2htai6d84...,SE:030:DayType:shtdvssbqnth2htai6d84vgj1pvajm1u,SE:030:OperatingPeriod:260509260613,NaT,NaN


,file,operating_period_id,from_date,to_date
0,_shared_data.xml,SE:030:OperatingPeriod:260620260809,2026-06-20,2026-08-10
1,_shared_data.xml,SE:030:OperatingPeriod:260706260807,2026-07-06,2026-08-08
2,_shared_data.xml,SE:030:OperatingPeriod:260530260530,2026-05-30,2026-05-31
3,_shared_data.xml,SE:030:OperatingPeriod:260507260618,2026-05-07,2026-06-19
4,_shared_data.xml,SE:030:OperatingPeriod:260509260613,2026-05-09,2026-06-14
5,_shared_data.xml,SE:030:OperatingPeriod:260516260516,2026-05-16,2026-05-17
6,_shared_data.xml,SE:030:OperatingPeriod:260619260815,2026-06-19,2026-08-16
7,_shared_data.xml,SE:030:OperatingPeriod:260510260614,2026-05-10,2026-06-15
8,_shared_data.xml,SE:030:OperatingPeriod:260615260617,2026-06-15,2026-06-18
9,_shared_data.xml,SE:030:OperatingPeriod:260507260611,2026-05-07,2026-06-12


## Comment

Both extractions completed successfully:

- **90,141 DayTypeAssignments** extracted from `_shared_data.xml`
- **2,595 OperatingPeriods** extracted, all with valid `from_date` and `to_date` values

The DayTypeAssignment sample shows two patterns, consistent with what the element
counts suggested earlier:
- Rows with an `operating_period_ref` and no `date` (these are date range assignments)
- Rows with a `date` and no `operating_period_ref` (these are direct single-date assignments)

The `is_available` column is `NaN` for all sampled rows, meaning Sweden does not use
this field. All assignments are therefore treated as active by default.

The first sampled OperatingPeriod rows fall within 2026, while the full
reconstructed NeTEx calendar later extends beyond 2026.

## Build NeTEx active dates

The NeTEx calendar data is stored as `DayTypeAssignment` elements that either
point to a specific date directly or to an `OperatingPeriod` date range. To
make the calendar comparable with GTFS, both sources are expanded into individual
active dates per `DayType`.

Since Sweden does not use the `IsAvailable` field, all assignments are treated
as active. The same approach as Norway is used, with two sources combined:

- **Direct date assignments** — each assignment contributes one active date

- **OperatingPeriod assignments** — each period is expanded into individual
  days between `from_date` and `to_date`

Duplicate dates for the same `DayType` are removed after combining both sources.

In [46]:
# Build NeTEx active dates from direct dates and OperatingPeriods

# All assignments are active since is_available is not used in Sweden
available_assignments = netex_daytype_assignments.copy()

# 1. Direct date assignments
netex_direct_dates = (
    available_assignments[
        available_assignments["date"].notna()
    ][["daytype_ref", "date"]]
    .copy()
    .rename(columns={"date": "active_date"})
)
netex_direct_dates["calendar_source"] = "direct date"

# 2. Assignments linked to OperatingPeriod
netex_period_assignments = available_assignments[
    available_assignments["date"].isna()
    & available_assignments["operating_period_ref"].notna()
].copy()

netex_period_assignments = netex_period_assignments.merge(
    netex_operating_periods[["operating_period_id", "from_date", "to_date"]],
    left_on="operating_period_ref",
    right_on="operating_period_id",
    how="left"
)

netex_period_assignments_valid = netex_period_assignments[
    netex_period_assignments["from_date"].notna()
    & netex_period_assignments["to_date"].notna()
].copy()

# Expand OperatingPeriods into individual active dates
period_date_rows = []
total = len(netex_period_assignments_valid)
for i, (_, row) in enumerate(netex_period_assignments_valid.iterrows(), start=1):
    print(f"  [{i}/{total}]", end="\r")
    for d in pd.date_range(row["from_date"], row["to_date"], freq="D"):
        period_date_rows.append({
            "daytype_ref":     row["daytype_ref"],
            "active_date":     d,
            "calendar_source": "operating period"
        })
print(f"\nDone — {len(period_date_rows)} period date rows expanded.")
netex_period_dates = pd.DataFrame(period_date_rows)

# 3. Combine both sources and deduplicate
netex_active_dates = pd.concat(
    [netex_direct_dates, netex_period_dates],
    ignore_index=True
).drop_duplicates(subset=["daytype_ref", "active_date"])

display(netex_active_dates.head(20))

  [7516/7516]
Done — 687289 period date rows expanded.


,daytype_ref,active_date,calendar_source
0,SE:030:DayType:utmge7pjhtmr530eqrpus3h7lciasaet,2026-06-20,direct date
1,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,2026-05-14,direct date
2,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,2026-06-15,direct date
3,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,2026-06-16,direct date
4,SE:030:DayType:2jett2pvjflsps8jvh15t2fv1i20tl1l,2026-06-17,direct date
5,SE:030:DayType:shtdvssbqnth2htai6d84vgj1pvajm1u,2026-05-16,direct date
6,SE:030:DayType:6gtlil4sfbvb98gq73fd98df4o9qk03t,2026-06-19,direct date
7,SE:030:DayType:6gtlil4sfbvb98gq73fd98df4o9qk03t,2026-06-20,direct date
8,SE:030:DayType:4nv825fibqs8lrs88hdoufiiokgp3bqd,2026-05-14,direct date
9,SE:030:DayType:n1631045ktjqvjk2s094238r0fb1ppo5,2026-05-14,direct date


## Comment

The active date expansion completed successfully:

- **687,289 period date rows** were expanded from 2,595 OperatingPeriods
- All sampled rows use **direct dates** as the calendar source, with dates
  falling in May and June 2026


## Prepare NeTEx calendar summary

The active dates table is aggregated into one summary row per `DayType`,
showing the number of active days, the first and last active date, and
whether the dates came from direct assignments, operating periods, or both.

In [47]:
# Prepare NeTEx calendar summary
netex_active_dates["active_date"] = pd.to_datetime(
    netex_active_dates["active_date"],
    errors="coerce"
).dt.normalize()

netex_calendar_summary = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .groupby("daytype_ref", as_index=False)
    .agg(
        n_active_days=("active_date", "nunique"),
        first_active_date=("active_date", "min"),
        last_active_date=("active_date", "max"),
        calendar_sources=("calendar_source", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .sort_values("daytype_ref")
    .reset_index(drop=True)
)

netex_calendar_summary_quality = pd.DataFrame({
    "metric": [
        "calendar summary rows",
        "daytype_ref with only 1 active day",
        "missing first_active_date",
        "missing last_active_date",
        "earliest first_active_date",
        "latest last_active_date"
    ],
    "value": [
        len(netex_calendar_summary),
        netex_calendar_summary["n_active_days"].eq(1).sum(),
        netex_calendar_summary["first_active_date"].isna().sum(),
        netex_calendar_summary["last_active_date"].isna().sum(),
        netex_calendar_summary["first_active_date"].min(),
        netex_calendar_summary["last_active_date"].max()
    ]
})

display(netex_calendar_summary.head(20))
display(netex_calendar_summary_quality)

,daytype_ref,n_active_days,first_active_date,last_active_date,calendar_sources
0,SE:030:DayType:001qqnjffnushqgvrgi7i62hf9lg9j9a,6,2026-06-22,2026-06-27,operating period
1,SE:030:DayType:008a5ft3kv149c7u3iv6ohtqkevqpn3u,9,2026-06-19,2026-06-27,operating period
2,SE:030:DayType:00c4rvgkru8v5fe2km07ua0tgo123q15,25,2026-05-29,2026-06-22,"direct date, operating period"
3,SE:030:DayType:00e01qgddin2o3c0n86o0s9g9q2db2uq,45,2026-05-16,2026-06-29,"direct date, operating period"
4,SE:030:DayType:00g0hhv4udt0onknmjg272fgn1ibub0v,9,2026-08-07,2026-08-15,operating period
5,SE:030:DayType:00kd0mulihadnj5vc6c3ct8goiqg0pis,8,2026-05-18,2026-05-25,operating period
6,SE:030:DayType:00vc3jqld6qq813v6kvenavco75juu54,170,2026-05-16,2026-11-01,"direct date, operating period"
7,SE:030:DayType:0173kld6k32cvno9b548nj750t6dei02,30,2026-07-17,2026-08-15,operating period
8,SE:030:DayType:01gm6olvu343dad29bmqqbk83n5e7uuf,2,2026-09-25,2026-09-26,operating period
9,SE:030:DayType:01mnrvv7ihn8nl6vbp9sclrarmm4pdbu,114,2026-05-11,2026-09-01,"direct date, operating period"


,metric,value
0,calendar summary rows,7516
1,daytype_ref with only 1 active day,0
2,missing first_active_date,0
3,missing last_active_date,0
4,earliest first_active_date,2024-12-15 00:00:00
5,latest last_active_date,2027-01-01 00:00:00


## Comment

The NeTEx calendar summary was created successfully with **7,516 DayType rows**,
matching exactly the number of `DayType` elements found in the structure check.

- No missing dates and no DayTypes with only 1 active day, confirming the
  extraction is complete and consistent
    
- The feed validity window runs from **2024-12-15 to 2027-01-01**
    
- Most DayTypes use either operating periods only or a combination of direct
  dates and operating periods, confirming the two-source pattern seen earlier
    
- Active day counts vary widely from 2 days (row 8) to 220 days (row 16),
  reflecting the mix of special services and year-round operations in the Swedish
  network


## Summary of NeTEx exploration

The Swedish NeTEx feed is structured differently from Norway but uses the same
Nordic NeTEx profile. Key findings:

**Stops:**
- 71,650 `StopPlace` elements extracted from `_stops.xml`, all with names
  and coordinates    
- `Quay` elements are nested inside `StopPlace` at depth 7 and are not used
  for the comparison    
- StopPlace IDs follow the format `SE:030:StopPlace:...` which is different from
  GTFS stop IDs, so the stop comparison will use coordinate proximity

**Lines:**
- 6,891 line files, one per line, all with a `PublicCode` and `TransportMode`
- After deduplication: 3,431 unique line labels

**Calendar:**
- A single `_shared_data.xml` file contains all calendar data
- Sweden uses `DayTypeAssignment` with direct dates and `OperatingPeriod`
  date ranges (no `ValidDayBits`)
- 7,516 DayTypes with a validity window from 2024-12-15 to 2027-01-01


# Part 3: GTFS–NeTEx Comparison

## 3.1 Station-level comparison



Unlike Norway, where both formats shared the same stop identifier, the Swedish
GTFS and NeTEx feeds use different ID systems. A direct ID match is therefore
not possible.

Instead, the comparison is done by **coordinate proximity**. For each GTFS stop,
the nearest NeTEx `StopPlace` is found using a spatial index (`cKDTree`). A match
is accepted if the nearest StopPlace is within **100 metres** of the GTFS stop.

This is the same approach used for the Netherlands, and is appropriate here because:
- The Swedish GTFS stop table is flat with no hierarchy
- NeTEx `StopPlace` elements represent the same concept as a GTFS stop (a named
  location with coordinates)
- Both datasets have complete coordinate coverage, making proximity matching reliable

The match rate is reported from both sides: the GTFS-side match rate shows how many
GTFS stops found a nearby NeTEx StopPlace, and the NeTEx-side coverage rate shows
what fraction of NeTEx StopPlaces were matched by at least one GTFS stop.

In [48]:
# Drop stops with missing coordinates on either side
gtfs_valid = stops.dropna(subset=["stop_lat", "stop_lon"]).copy()
netex_valid = netex_stopplaces.dropna(subset=["netex_lat", "netex_lon"]).copy()

gtfs_valid["stop_lat"] = gtfs_valid["stop_lat"].astype(float)
gtfs_valid["stop_lon"] = gtfs_valid["stop_lon"].astype(float)

# Build cKDTree from NeTEx coordinates
netex_array = np.array([[row.netex_lat, row.netex_lon] for _, row in netex_valid.iterrows()])
gtfs_array  = np.array([[row.stop_lat,  row.stop_lon]  for _, row in gtfs_valid.iterrows()])

tree = cKDTree(netex_array)
distances, indices = tree.query(gtfs_array, k=1)

# Convert degrees to metres
distances_m = distances * 111000

# Attach results to GTFS stops
gtfs_valid = gtfs_valid.copy()
gtfs_valid["nearest_netex_id"]   = netex_valid.iloc[indices]["netex_stopplace_id"].values
gtfs_valid["nearest_netex_name"] = netex_valid.iloc[indices]["netex_stopplace_name"].values
gtfs_valid["distance_m"]         = distances_m

# Apply match threshold
THRESHOLD_M = 100
gtfs_valid["matched"] = gtfs_valid["distance_m"] <= THRESHOLD_M

station_match_summary = pd.DataFrame({
    "metric": [
        "GTFS stops",
        "NeTEx StopPlaces",
        "GTFS stops matched (within 100m)",
        "GTFS stops unmatched",
        "GTFS-side match rate (%)",
        "NeTEx-side coverage rate (%)"
    ],
    "value": [
        len(gtfs_valid),
        len(netex_valid),
        gtfs_valid["matched"].sum(),
        (~gtfs_valid["matched"]).sum(),
        round(gtfs_valid["matched"].mean() * 100, 2),
        round(gtfs_valid["matched"].sum() / len(netex_valid) * 100, 2)
    ]
})
display(station_match_summary)
display(gtfs_valid.sort_values("distance_m", ascending=False).head(20))

,metric,value
0,GTFS stops,48778.00
1,NeTEx StopPlaces,71650.00
2,GTFS stops matched (within 100m),48137.00
3,GTFS stops unmatched,641.00
4,GTFS-side match rate (%),98.69
5,NeTEx-side coverage rate (%),67.18


,stop_id,stop_name,stop_lat,stop_lon,location_type,nearest_netex_id,nearest_netex_name,distance_m,matched
11,510005320,Wroclaw Dworzec autobusowy,53.615125,17.597223,NaN,SE:030:StopPlace:9021050073196000,"Poznan, Bus Station",154473.595489,False
48755,860000053,Århus H,56.307945,10.626952,NaN,SE:030:StopPlace:9021050073043000,Aarhus C FlixBus stop,49546.148450,False
48717,780089051,Zagreb Autobusni Kolodvor,47.063665,11.758251,NaN,SE:030:StopPlace:9021050072906000,Jenbach,36103.139586,False
48725,800010140,Berlin flughafen BER T1/2,52.422932,13.669689,NaN,SE:030:StopPlace:9021050073002000,Schönefeld (bei Berlin),18104.252563,False
48750,800099992,Hamburg Airport,53.726455,9.986629,NaN,SE:030:StopPlace:9021050073068000,Hamburg Airport,10649.112706,False
9359,740018287,Torrflonäs,62.325825,15.126094,NaN,SE:030:StopPlace:9021050066182000,Säter Västra,6821.824098,False
9341,740018264,Tängvattnet,65.832831,14.852524,NaN,SE:030:StopPlace:9021050072869000,Tängvattnet,5418.723021,False
15284,740027280,Norra Prästholm,65.910983,22.199743,NaN,SE:030:StopPlace:9021050057782000,N Prästholm,4832.231373,False
12916,740023722,Svarttjärn,64.625372,21.024384,NaN,SE:030:StopPlace:9021050072421000,Skellefteå flygplats,4599.447117,False
14515,740026021,Arvidsjaur flygplats,65.592259,19.263183,NaN,SE:030:StopPlace:9021050057047000,Arvidsjaurs Flygplats,4550.821347,False


In [49]:
print("Example matched stops:")
display(gtfs_valid[gtfs_valid["matched"]].sort_values("distance_m").head(20))

Example matched stops:


,stop_id,stop_name,stop_lat,stop_lon,location_type,nearest_netex_id,nearest_netex_name,distance_m,matched
46769,740076175,Bångåsen Gunnebo,63.480977,13.972053,NaN,SE:030:StopPlace:9021050058976000,Bångåsen Gunnebo,0.0,True
3151,740008601,Kölinge hembygdsgård,60.027750,17.874999,NaN,SE:030:StopPlace:9021050005075000,Kölinge hembygdsgård (Uppsala),0.0,True
3146,740008594,Börje skola,59.883791,17.505581,NaN,SE:030:StopPlace:9021050005066000_1,Börje skola (Uppsala),0.0,True
28005,740047262,Uppfartsvägen,60.521640,15.437356,NaN,SE:030:StopPlace:9021050045351000,Uppfartsvägen,0.0,True
28003,740047260,Haggården Borlänge,60.514676,15.440840,NaN,SE:030:StopPlace:9021050045353000,Haggården Borlänge,0.0,True
3174,740008653,Ala,59.691296,17.713434,NaN,SE:030:StopPlace:9021050003290000,Ala (Knivsta),0.0,True
3172,740008637,Skäve,59.890486,17.698755,NaN,SE:030:StopPlace:9021050005112000,Skäve (Uppsala),0.0,True
3167,740008631,Uppsala Viktoria,59.846173,17.711583,NaN,SE:030:StopPlace:9021050005106000,Viktoria (Uppsala),0.0,True
27996,740047252,Domnarvets skola,60.501197,15.454748,NaN,SE:030:StopPlace:9021050047300000,Domnarvets skola,0.0,True
27995,740047251,Domnarvet centrum,60.499925,15.455672,NaN,SE:030:StopPlace:9021050045360000,Domnarvet centrum,0.0,True


## Comment

The coordinate-based stop comparison gives a strong result overall, with a few
interesting observations:

The **GTFS-side match rate is 98.69%**. 48,137 out of 48,778 GTFS stops found
a NeTEx StopPlace within 100 metres. Only 641 stops were unmatched.

The **NeTEx-side coverage rate is 67.18%**, meaning NeTEx contains significantly
more StopPlaces than GTFS covers. This is consistent with the stop counts (71,650 NeTEx StopPlaces vs 48,778 GTFS stops) suggesting NeTEx includes stops
that are not represented in the GTFS feed.

Distance is approximated using a flat 111,000 m/degree conversion; this slightly
overestimates true distance at Sweden's latitude for longitude-dominant offsets,
so the reported match rate is a conservative lower bound.

The unmatched GTFS stops shown in the table reveal two distinct patterns:

- **International stops** — Wroclaw, Zagreb, Aarhus, Berlin, Hamburg Airport
  are stops outside Sweden entirely, likely from international coach services.
  These have no nearby Swedish NeTEx StopPlace by definition and will always
  be unmatched.
- **Remote or rural Swedish stops** — stops like Torrflonäs, Tängvattnet,
  Norra Prästholm appear to exist in GTFS but have no matching NeTEx StopPlace
  nearby, suggesting incomplete NeTEx coverage in remote areas.

Overall the stop-level MVD is **strongly confirmed** for Sweden, with the
caution that international stops in GTFS are structurally unmatchable against
a national NeTEx feed.

## 3.2 Line-level comparison

The GTFS routes are first deduplicated by public label into one row per unique
label. The approach is the following: `route_short_name` is used as the
primary label, with `route_long_name` as fallback for routes where
`route_short_name` is missing.

Both the GTFS and NeTEx label sets are then compared using set operations, 
labels that appear in both datasets are counted as matches, while labels only
in GTFS or only in NeTEx are counted separately. Labels are kept as plain text
and stripped of spaces to avoid false mismatches from formatting differences.

In [50]:
# Deduplicate GTFS routes by public label
gtfs_lines_dedup = (
    routes_check[routes_check["gtfs_public_label"].notna()]
    .groupby("gtfs_public_label", as_index=False)
    .agg(
        number_of_route_rows=("route_id", "count"),
        route_types=("route_type", lambda x: ", ".join(map(str, sorted(x.dropna().unique())))),
        example_route_long_name=("route_long_name", "first")
    )
    .rename(columns={"gtfs_public_label": "gtfs_line_label"})
    .sort_values(
        "gtfs_line_label",
        key=lambda s: pd.to_numeric(s, errors="coerce").fillna(float("inf"))
    )
    .reset_index(drop=True)
)

# Line-level GTFS–NeTEx comparison by public label
gtfs_line_compare  = gtfs_lines_dedup.copy()
netex_line_compare = netex_lines_dedup.copy()

gtfs_line_compare["line_label_compare"]  = gtfs_line_compare["gtfs_line_label"].astype(str).str.strip()
netex_line_compare["line_label_compare"] = netex_line_compare["netex_line_label"].astype(str).str.strip()

gtfs_labels  = set(gtfs_line_compare["line_label_compare"])
netex_labels = set(netex_line_compare["line_label_compare"])

matched_labels    = gtfs_labels & netex_labels
gtfs_only_labels  = gtfs_labels - netex_labels
netex_only_labels = netex_labels - gtfs_labels

gtfs_match_rate  = round(len(matched_labels) / len(gtfs_labels)  * 100, 2) if gtfs_labels  else 0.0
netex_match_rate = round(len(matched_labels) / len(netex_labels) * 100, 2) if netex_labels else 0.0

line_match_summary = pd.DataFrame({
    "metric": [
        "GTFS unique line labels",
        "NeTEx unique public codes",
        "matched labels",
        "GTFS-only labels",
        "NeTEx-only labels",
        "GTFS-side match rate (%)",
        "NeTEx-side match rate (%)"
    ],
    "value": [
        len(gtfs_labels),
        len(netex_labels),
        len(matched_labels),
        len(gtfs_only_labels),
        len(netex_only_labels),
        gtfs_match_rate,
        netex_match_rate
    ]
})
display(line_match_summary)

print("Example matched labels:")
display(gtfs_line_compare[gtfs_line_compare["line_label_compare"].isin(matched_labels)].head(20))
print("Example GTFS-only labels:")
display(gtfs_line_compare[gtfs_line_compare["line_label_compare"].isin(gtfs_only_labels)].head(20))
print("Example NeTEx-only labels:")
display(netex_line_compare[netex_line_compare["line_label_compare"].isin(netex_only_labels)].head(20))

,metric,value
0,GTFS unique line labels,2904.00
1,NeTEx unique public codes,3431.00
2,matched labels,1580.00
3,GTFS-only labels,1324.00
4,NeTEx-only labels,1851.00
5,GTFS-side match rate (%),54.41
6,NeTEx-side match rate (%),46.05


Example matched labels:


,gtfs_line_label,number_of_route_rows,route_types,example_route_long_name,line_label_compare
0,1,67,"700, 900, 1000, 1501",None,1
1,1.,1,700,None,1.
2,2,69,"700, 900, 1000, 1501",None,2
3,2.,1,700,None,2.
4,3.,1,700,None,3.
5,3,56,"700, 900, 1000",None,3
6,4,43,"700, 900, 1000, 1501",None,4
7,4.,1,700,None,4.
8,5,31,"700, 900, 1000",None,5
9,5.,1,700,None,5.


Example GTFS-only labels:


,gtfs_line_label,number_of_route_rows,route_types,example_route_long_name,line_label_compare
371,383,1,101,None,383
374,386,1,101,None,386
376,388,1,101,None,388
377,389,1,101,None,389
386,398,1,101,None,398
441,454,1,700,None,454
846,888,1,702,None,888
962,2003,1,101,None,2003
963,2004,1,101,None,2004
964,2006,1,101,None,2006


Example NeTEx-only labels:


,netex_line_label,number_of_line_rows,transport_modes,example_line_name,route_count_total,service_journey_count_total,line_label_compare
192,187,1,rail,187,4,14,187
194,189,1,rail,189,13,47,189
262,257,1,rail,257,2,8,257
264,259,1,rail,259,4,6,259
277,272,1,rail,272,4,6,272
283,278,1,rail,278,4,29,278
284,279,1,rail,279,6,17,279
285,280,1,rail,280,5,10,280
293,288,1,rail,288,4,8,288
331,327,1,rail,327,4,26,327


## Comment

The line-level comparison gives a moderate result for Sweden, noticeably lower
than Norway's near-perfect match.

- **2,904 unique GTFS labels** and **3,431 unique NeTEx labels**
- **1,580 matched** &rarr; a GTFS-side match rate of **54.41%** and NeTEx-side of **46.05%**
- **1,324 GTFS-only labels** and **1,851 NeTEx-only labels**

The lower match rate compared to Norway is largely explained by two structural
issues visible in the output:

**Trailing dot labels** — NeTEx contains labels like "1.", "2.", "3." alongside
"1", "2", "3". These are treated as different labels by the set comparison even
though they likely refer to the same line. This alone accounts for a significant
number of mismatches.

**GTFS-only labels** — many are large numeric values like 383, 386, 388, 2003,
2004, mostly with route type 101 (long-distance rail). These appear to be
regional rail line numbers that exist in GTFS Sverige 2 but are not present
in the NeTEx feed, consistent with the known incomplete operator coverage of
NeTEx Sweden.

**NeTEx-only labels** — mostly rail and bus lines with numeric labels like 187,
189, 257 that exist in NeTEx but not in GTFS, again reflecting the different
operator coverage between the two feeds.

Overall the line-level MVD is **partially confirmed** for Sweden. The data needed
to identify lines is present in both formats, but the match rate is lower than
Norway due to label formatting inconsistencies and different operator coverage
between the two feeds.

## 3.3 Calendar-level comparison

Before comparing calendars, the GTFS active dates are converted into a dictionary
mapping each `service_id` to its set of active dates. Since the Swedish GTFS feed
does not use weekly patterns in `calendar.txt` (all weekday flags are set to 0)
the active dates come entirely from `calendar_dates.txt`, using only rows where
`exception_type = 1` (added dates).

This dictionary is the starting point for the calendar comparison, which follows
the same approach as Norway: ID overlap check, active date comparison for matched
IDs, and pattern-level comparison over a shared date window.

In [51]:
# Build GTFS active dates dictionary from calendar_dates_check
gtfs_active_dates = (
    calendar_dates_check[calendar_dates_check["exception_type"] == 1]
    .groupby("service_id")["date"]
    .apply(set)
    .to_dict()
)

print(f"GTFS active dates built for {len(gtfs_active_dates)} service IDs.")

GTFS active dates built for 6892 service IDs.


## Calendar ID overlap check

The first step of the calendar comparison checks whether any GTFS `service_id`
values also appear as NeTEx `DayType` references. Unlike stations and lines,
calendar IDs are not guaranteed to overlap, each format may use its own internal
naming convention.

This check gives a first indication of how closely the two formats are aligned
at the identifier level, before moving to the active date comparison.

In [52]:
# Calendar-level ID coverage between GTFS service_id and NeTEx daytype_ref
gtfs_calendar_ids  = set(gtfs_calendar_summary["service_id"].dropna())
netex_calendar_ids = set(netex_calendar_summary["daytype_ref"].dropna())

matched_calendar_ids    = gtfs_calendar_ids & netex_calendar_ids
gtfs_only_calendar_ids  = gtfs_calendar_ids - netex_calendar_ids
netex_only_calendar_ids = netex_calendar_ids - gtfs_calendar_ids

calendar_id_summary = pd.DataFrame({
    "metric": [
        "GTFS service_id values",
        "NeTEx daytype_ref values",
        "matched calendar IDs",
        "GTFS-only calendar IDs",
        "NeTEx-only calendar IDs",
        "GTFS-side ID match rate (%)",
        "NeTEx-side ID match rate (%)"
    ],
    "value": [
        len(gtfs_calendar_ids),
        len(netex_calendar_ids),
        len(matched_calendar_ids),
        len(gtfs_only_calendar_ids),
        len(netex_only_calendar_ids),
        round(len(matched_calendar_ids) / len(gtfs_calendar_ids)  * 100, 2) if gtfs_calendar_ids  else 0.0,
        round(len(matched_calendar_ids) / len(netex_calendar_ids) * 100, 2) if netex_calendar_ids else 0.0
    ]
})
display(calendar_id_summary)

print("Example matched calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(matched_calendar_ids))[:20]}))
print("Example GTFS-only calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(gtfs_only_calendar_ids))[:20]}))
print("Example NeTEx-only calendar IDs:")
display(pd.DataFrame({"calendar_id": sorted(list(netex_only_calendar_ids))[:20]}))

,metric,value
0,GTFS service_id values,6892.0
1,NeTEx daytype_ref values,7516.0
2,matched calendar IDs,0.0
3,GTFS-only calendar IDs,6892.0
4,NeTEx-only calendar IDs,7516.0
5,GTFS-side ID match rate (%),0.0
6,NeTEx-side ID match rate (%),0.0


Example matched calendar IDs:


,calendar_id


Example GTFS-only calendar IDs:


,calendar_id
0,0
1,1
2,100
3,1000
4,1001
5,1003
6,1004
7,1005
8,1007
9,1009


Example NeTEx-only calendar IDs:


,calendar_id
0,SE:030:DayType:001qqnjffnushqgvrgi7i62hf9lg9j9a
1,SE:030:DayType:008a5ft3kv149c7u3iv6ohtqkevqpn3u
2,SE:030:DayType:00c4rvgkru8v5fe2km07ua0tgo123q15
3,SE:030:DayType:00e01qgddin2o3c0n86o0s9g9q2db2uq
4,SE:030:DayType:00g0hhv4udt0onknmjg272fgn1ibub0v
5,SE:030:DayType:00kd0mulihadnj5vc6c3ct8goiqg0pis
6,SE:030:DayType:00vc3jqld6qq813v6kvenavco75juu54
7,SE:030:DayType:0173kld6k32cvno9b548nj750t6dei02
8,SE:030:DayType:01gm6olvu343dad29bmqqbk83n5e7uuf
9,SE:030:DayType:01mnrvv7ihn8nl6vbp9sclrarmm4pdbu


## Comment

The ID overlap check shows **0 matched calendar IDs**. This is completely different from Norway, where 67% of GTFS service IDs
matched a NeTEx DayType reference.

The reason is clear from the example IDs:

- **GTFS-only IDs** are plain numeric values like 0, 1, 100
- **NeTEx-only IDs** follow another format like `SE:030:DayType:...` 

This means the calendar comparison cannot rely on ID matching at all for Sweden.
The only viable approach is the **pattern-level comparison** over a shared date
window, which compares the actual active dates directly without requiring any
ID overlap.

## Pattern-level calendar comparison

Since the GTFS and NeTEx feeds use completely different calendar ID systems with
no overlap, a direct ID-based comparison is not possible for Sweden. The calendar
comparison is therefore done entirely at the **pattern level** comparing the
actual active dates directly without relying on shared identifiers.

The first step defines a shared date window: the period covered by both feeds
simultaneously. All active dates outside this window are excluded, so that both
sides are compared over exactly the same time range.

In [53]:
# Normalize GTFS active dates
gtfs_dates_by_id = {
    service_id: set(pd.to_datetime(list(dates)).normalize())
    for service_id, dates in gtfs_active_dates.items()
}

# Normalize NeTEx active dates
netex_dates_by_id = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .assign(active_date=lambda df: pd.to_datetime(df["active_date"]).dt.normalize())
    .groupby("daytype_ref")["active_date"]
    .apply(set)
    .to_dict()
)

# Define shared comparison window
all_gtfs_dates  = [d for dates in gtfs_dates_by_id.values()  for d in dates]
all_netex_dates = [d for dates in netex_dates_by_id.values() for d in dates]

gtfs_min_date  = min(all_gtfs_dates)
gtfs_max_date  = max(all_gtfs_dates)
netex_min_date = min(all_netex_dates)
netex_max_date = max(all_netex_dates)

shared_start_date = max(gtfs_min_date, netex_min_date)
shared_end_date   = min(gtfs_max_date, netex_max_date)
shared_dates      = pd.date_range(shared_start_date, shared_end_date, freq="D")

shared_window_summary = pd.DataFrame({
    "metric": [
        "GTFS earliest active date",
        "GTFS latest active date",
        "NeTEx earliest active date",
        "NeTEx latest active date",
        "shared start date",
        "shared end date",
        "number of days in shared window"
    ],
    "value": [
        gtfs_min_date,
        gtfs_max_date,
        netex_min_date,
        netex_max_date,
        shared_start_date,
        shared_end_date,
        len(shared_dates)
    ]
})
display(shared_window_summary)

,metric,value
0,GTFS earliest active date,2026-05-01 00:00:00
1,GTFS latest active date,2027-04-30 00:00:00
2,NeTEx earliest active date,2024-12-15 00:00:00
3,NeTEx latest active date,2027-01-01 00:00:00
4,shared start date,2026-05-01 00:00:00
5,shared end date,2027-01-01 00:00:00
6,number of days in shared window,246


## Comment

The shared date window runs from **2026-05-01 to 2027-01-01**, covering **246 days**.

- The GTFS feed covers exactly one year: 2026-05-01 to 2027-04-30
- The NeTEx feed has a much wider window: 2024-12-15 to 2027-01-01, spanning
  over two years
- The shared window is limited by the GTFS start date on one side and the NeTEx
  end date on the other

The 246-day shared window is reasonably long and covers a full seasonal cycle
including summer, autumn and winter services, which should give a representative
basis for the pattern comparison.

## Build and compare calendar activity patterns

Since no calendar IDs are shared between GTFS and NeTEx, the comparison is done
by converting each service's active dates into a binary activity pattern where each position represents one day in the shared window.

Two pattern tables are built, one for GTFS and one for NeTEx. For each service,
the active dates are filtered to the shared window and encoded as a bit string.
Services with no active dates in the shared window are excluded.

The two pattern sets are then compared to see how many patterns appear in both
formats, regardless of which ID they carry.

In [54]:
# Build activity patterns over the shared comparison window
def build_activity_pattern(active_dates, comparison_dates):
    active_dates = set(active_dates)
    return "".join("1" if d in active_dates else "0" for d in comparison_dates)

# Build GTFS calendar patterns
gtfs_pattern_rows = []
total_gtfs = len(gtfs_dates_by_id)
for i, (service_id, active_dates) in enumerate(gtfs_dates_by_id.items(), start=1):
    print(f"  GTFS [{i}/{total_gtfs}]", end="\r")
    window_dates = {
        d for d in active_dates
        if shared_start_date <= d <= shared_end_date
    }
    gtfs_pattern_rows.append({
        "gtfs_service_id":          service_id,
        "n_active_days_window":     len(window_dates),
        "first_active_date_window": min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window":  max(window_dates) if window_dates else pd.NaT,
        "activity_pattern":         build_activity_pattern(window_dates, shared_dates)
    })
print(f"  GTFS done — {total_gtfs} calendars processed.")
gtfs_calendar_patterns = pd.DataFrame(gtfs_pattern_rows)

# Build NeTEx calendar patterns
netex_pattern_rows = []
total_netex = len(netex_dates_by_id)
for i, (daytype_ref, active_dates) in enumerate(netex_dates_by_id.items(), start=1):
    print(f"  NeTEx [{i}/{total_netex}]", end="\r")
    window_dates = {
        d for d in active_dates
        if shared_start_date <= d <= shared_end_date
    }
    netex_pattern_rows.append({
        "netex_daytype_ref":         daytype_ref,
        "n_active_days_window":      len(window_dates),
        "first_active_date_window":  min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window":   max(window_dates) if window_dates else pd.NaT,
        "activity_pattern":          build_activity_pattern(window_dates, shared_dates)
    })
print(f"  NeTEx done — {total_netex} calendars processed.")
netex_calendar_patterns = pd.DataFrame(netex_pattern_rows)

# Keep only calendars with at least one active date in the shared window
gtfs_calendar_patterns_active  = gtfs_calendar_patterns[
    gtfs_calendar_patterns["n_active_days_window"] > 0
].copy()
netex_calendar_patterns_active = netex_calendar_patterns[
    netex_calendar_patterns["n_active_days_window"] > 0
].copy()

pattern_preparation_summary = pd.DataFrame({
    "metric": [
        "GTFS calendar rows",
        "GTFS rows with active days in shared window",
        "GTFS rows with zero active days in shared window",
        "GTFS unique activity patterns",
        "NeTEx calendar rows",
        "NeTEx rows with active days in shared window",
        "NeTEx rows with zero active days in shared window",
        "NeTEx unique activity patterns"
    ],
    "value": [
        len(gtfs_calendar_patterns),
        len(gtfs_calendar_patterns_active),
        (gtfs_calendar_patterns["n_active_days_window"] == 0).sum(),
        gtfs_calendar_patterns_active["activity_pattern"].nunique(),
        len(netex_calendar_patterns),
        len(netex_calendar_patterns_active),
        (netex_calendar_patterns["n_active_days_window"] == 0).sum(),
        netex_calendar_patterns_active["activity_pattern"].nunique()
    ]
})
display(pattern_preparation_summary)

print("Example GTFS calendar patterns:")
display(gtfs_calendar_patterns_active.head(10))
print("Example NeTEx calendar patterns:")
display(netex_calendar_patterns_active.head(10))

  GTFS done — 6892 calendars processed.
  NeTEx done — 7516 calendars processed.


,metric,value
0,GTFS calendar rows,6892
1,GTFS rows with active days in shared window,6892
2,GTFS rows with zero active days in shared window,0
3,GTFS unique activity patterns,6892
4,NeTEx calendar rows,7516
5,NeTEx rows with active days in shared window,7484
6,NeTEx rows with zero active days in shared window,32
7,NeTEx unique activity patterns,2563


Example GTFS calendar patterns:


,gtfs_service_id,n_active_days_window,first_active_date_window,last_active_date_window,activity_pattern
0,0,246,2026-05-01,2027-01-01,1111111111111111111111111111111111111111111111...
1,1,5,2026-05-09,2026-06-13,0000000010000001000000100000010000000000000100...
2,3,40,2026-06-22,2026-08-14,0000000000000000000000000000000000000000000000...
3,5,9,2026-06-20,2026-08-09,0000000000000000000000000000000000000000000000...
4,7,32,2026-05-05,2026-06-18,0000111100111010011111001111100111110011111001...
5,8,45,2026-05-05,2026-06-18,0000111111111111111111111111111111111111111111...
6,9,13,2026-05-09,2026-06-14,0000000011000101100000110000011000001100000110...
7,11,18,2026-06-19,2026-08-15,0000000000000000000000000000000000000000000000...
8,12,58,2026-06-19,2026-08-15,0000000000000000000000000000000000000000000000...
9,13,8,2026-05-10,2026-06-14,0000000001000100100000010000001000001100000010...


Example NeTEx calendar patterns:


,netex_daytype_ref,n_active_days_window,first_active_date_window,last_active_date_window,activity_pattern
0,SE:030:DayType:001qqnjffnushqgvrgi7i62hf9lg9j9a,6,2026-06-22,2026-06-27,0000000000000000000000000000000000000000000000...
1,SE:030:DayType:008a5ft3kv149c7u3iv6ohtqkevqpn3u,9,2026-06-19,2026-06-27,0000000000000000000000000000000000000000000000...
2,SE:030:DayType:00c4rvgkru8v5fe2km07ua0tgo123q15,25,2026-05-29,2026-06-22,0000000000000000000000000000111111111111111111...
3,SE:030:DayType:00e01qgddin2o3c0n86o0s9g9q2db2uq,45,2026-05-16,2026-06-29,0000000000000001111111111111111111111111111111...
4,SE:030:DayType:00g0hhv4udt0onknmjg272fgn1ibub0v,9,2026-08-07,2026-08-15,0000000000000000000000000000000000000000000000...
5,SE:030:DayType:00kd0mulihadnj5vc6c3ct8goiqg0pis,8,2026-05-18,2026-05-25,0000000000000000011111111000000000000000000000...
6,SE:030:DayType:00vc3jqld6qq813v6kvenavco75juu54,170,2026-05-16,2026-11-01,0000000000000001111111111111111111111111111111...
7,SE:030:DayType:0173kld6k32cvno9b548nj750t6dei02,30,2026-07-17,2026-08-15,0000000000000000000000000000000000000000000000...
8,SE:030:DayType:01gm6olvu343dad29bmqqbk83n5e7uuf,2,2026-09-25,2026-09-26,0000000000000000000000000000000000000000000000...
9,SE:030:DayType:01mnrvv7ihn8nl6vbp9sclrarmm4pdbu,114,2026-05-11,2026-09-01,0000000000111111111111111111111111111111111111...


## Comment

Both pattern tables were built successfully:

- All 6,892 GTFS service IDs have active days in the shared window, producing
  6,892 unique activity patterns (every single GTFS service has a distinct pattern)
- 7,484 out of 7,516 NeTEx DayTypes have active days in the shared window, with
  32 falling entirely outside it. These 32 are excluded from the comparison
- NeTEx produces 2,563 unique patterns, significantly fewer than GTFS

The activity patterns in both samples look plausible. Service ID 0 in GTFS runs
every single day of the window (246 ones), while others show sparse or seasonal
patterns as expected.

The large number of unique GTFS patterns (6,892) compared to NeTEx (2,563) suggests
that Sweden's GTFS feed models calendar at a much finer granularity than NeTEx,
with almost every service having its own distinct pattern.

## Compare unique calendar activity patterns

The two pattern sets are now compared directly. Each unique activity pattern is
treated as a fingerprint of a calendar so if the same pattern appears in both
GTFS and NeTEx, it means both formats describe at least one service that runs
on exactly the same days, even if the IDs are different.

The match rate is reported from both sides: the GTFS-side rate shows how many
distinct GTFS patterns also appear in NeTEx, and the NeTEx-side rate shows how
many NeTEx patterns also appear in GTFS.

In [55]:
# Compare unique GTFS and NeTEx activity patterns
gtfs_unique_patterns = (
    gtfs_calendar_patterns_active
    .assign(pattern_active_days=lambda df: df["activity_pattern"].str.count("1"))
    .groupby("activity_pattern", as_index=False)
    .agg(
        number_of_gtfs_ids=("gtfs_service_id", "count"),
        example_gtfs_service_id=("gtfs_service_id", "first"),
        n_active_days_window=("pattern_active_days", "first"),
        first_active_date_window=("first_active_date_window", "first"),
        last_active_date_window=("last_active_date_window", "first")
    )
)

netex_unique_patterns = (
    netex_calendar_patterns_active
    .assign(pattern_active_days=lambda df: df["activity_pattern"].str.count("1"))
    .groupby("activity_pattern", as_index=False)
    .agg(
        number_of_netex_ids=("netex_daytype_ref", "count"),
        example_netex_daytype_ref=("netex_daytype_ref", "first"),
        n_active_days_window=("pattern_active_days", "first"),
        first_active_date_window=("first_active_date_window", "first"),
        last_active_date_window=("last_active_date_window", "first")
    )
)

gtfs_pattern_set   = set(gtfs_unique_patterns["activity_pattern"])
netex_pattern_set  = set(netex_unique_patterns["activity_pattern"])
matched_patterns   = gtfs_pattern_set & netex_pattern_set
gtfs_only_patterns = gtfs_pattern_set - netex_pattern_set
netex_only_patterns = netex_pattern_set - gtfs_pattern_set

gtfs_pattern_match_rate  = round(len(matched_patterns) / len(gtfs_pattern_set)  * 100, 2) if gtfs_pattern_set  else 0.0
netex_pattern_match_rate = round(len(matched_patterns) / len(netex_pattern_set) * 100, 2) if netex_pattern_set else 0.0

pattern_match_summary = pd.DataFrame({
    "metric": [
        "GTFS unique activity patterns",
        "NeTEx unique activity patterns",
        "matched activity patterns",
        "GTFS-only activity patterns",
        "NeTEx-only activity patterns",
        "GTFS-side pattern match rate (%)",
        "NeTEx-side pattern match rate (%)"
    ],
    "value": [
        len(gtfs_pattern_set),
        len(netex_pattern_set),
        len(matched_patterns),
        len(gtfs_only_patterns),
        len(netex_only_patterns),
        gtfs_pattern_match_rate,
        netex_pattern_match_rate
    ]
})
display(pattern_match_summary)

,metric,value
0,GTFS unique activity patterns,6892.00
1,NeTEx unique activity patterns,2563.00
2,matched activity patterns,475.00
3,GTFS-only activity patterns,6417.00
4,NeTEx-only activity patterns,2088.00
5,GTFS-side pattern match rate (%),6.89
6,NeTEx-side pattern match rate (%),18.53


## Comment

The pattern-level comparison gives the lowest match rate seen across all countries
analysed so far:

- **475 patterns matched** out of 6,892 GTFS and 2,563 NeTEx patterns
- **GTFS-side match rate: 6.89%** &rarr; only 1 in 14 GTFS patterns has an exact
  equivalent in NeTEx
- **NeTEx-side match rate: 18.53%** &rarr; about 1 in 5 NeTEx patterns appears in GTFS

The very low GTFS-side rate is largely explained by the extreme granularity of
the Swedish GTFS feed. 6,892 unique patterns for 6,892 service IDs means every
single service has a completely distinct calendar. This level of granularity is
unlikely to be replicated exactly in NeTEx, which uses only 2,563 patterns for
7,516 DayTypes, suggesting a more aggregated calendar model.

This structural difference, GTFS modelling one pattern per service, NeTEx
grouping multiple services under shared DayTypes, is the main driver of the
low match rate, rather than genuinely different service validity data.

## Calendar comparison: approach and results

Since the GTFS and NeTEx feeds in Sweden use completely different calendar ID
systems with no overlap, the comparison was done in two steps.

**Step 1 — ID overlap check**

The first step checked whether any GTFS `service_id` values also appear as NeTEx
`DayType` references. Out of 6,892 GTFS service IDs and 7,516 NeTEx DayType
references, **0 IDs matched**. 

**Step 2 — Pattern-level comparison over a shared date window**

Since no IDs could be matched, the comparison moved directly to the pattern level.
Each service's active dates were encoded as a binary string over the shared date
window of **246 days** (2026-05-01 to 2027-01-01). Two services are considered
equivalent if their bit strings are identical, meaning they run on exactly the
same days.

GTFS produced **6,892 unique patterns** one per service ID, meaning every GTFS
service has a completely distinct calendar. NeTEx produced **2,563 unique patterns**
for 7,516 DayTypes, reflecting a more aggregated calendar model where multiple
services share the same DayType.

Of these, **475 patterns matched**, giving a GTFS-side match rate of **6.89%** and
a NeTEx-side match rate of **18.53%**.

The low match rate is primarily a structural difference rather than missing data.
Both formats contain the calendar information needed to reconstruct when services
run, but they organise it at very different levels of granularity. 

## Overall summary


**Stops**

98.69% of GTFS stops matched a NeTEx StopPlace within 100 metres. The 641
unmatched stops are mostly international coach stops located outside Sweden,
which have no counterpart in the Swedish NeTEx feed by definition. For domestic
stops, the coverage is effectively complete.

**Lines**

54.41% of GTFS line labels matched a NeTEx public code. The lower rate compared
to Norway is mainly explained by different operator coverage between the two feeds.
GTFS Sverige 2 covers all operators nationally while NeTEx Sweden does not yet
cover all operators.

**Calendar**

The two feeds use completely different calendar ID systems with 0% ID overlap.
The pattern-level comparison over the shared window gave a GTFS-side match rate
of 6.89% and a NeTEx-side rate of 18.53%. 
